# Dot product: a two-stage reduction

A reduction turns $n$ values into one. Threads in **different blocks** cannot synchronise inside a kernel, so a grid-wide reduction is split into stages:

1. **Stage 1** (`BLOCKS = 8` blocks of `TPB = 64` threads): each thread accumulates $a_i b_i$ over a grid-stride loop, each block tree-reduces its 64 partials in shared memory, and thread 0 writes the block's total to `partials[blockIdx.x]`.
2. **Stage 2** (1 block of 64 threads): reduce the 8 partials the same way and write the final value to a 1-element device array.

Kernel launches on the same stream run in order, so stage 2 sees everything stage 1 wrote. No host round trip in between.

## Task
Write `dot(a, b)` for two `float32` arrays of length `n` (`n` may be 0) and return $\sum_i a_i b_i$ as a Python `float` (0.0 for empty inputs).

- Accumulate in `float32`. Summation order differs between implementations, so results are compared with relative and absolute tolerance $10^{-4}$.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import numpy as np
from numba import cuda, float32

BLOCKS, TPB = 8, 64

def dot(a, b):
    """a, b: float32 (n,). Returns sum(a * b) as a Python float."""
    raise NotImplementedError

@cuda.jit
def partial_dot_kernel(a, b, partials):
    pass

@cuda.jit
def final_sum_kernel(partials, out):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [1.0, 2.0, 3.0], "dtype": "float32"}, {"__nd__": [4.0, 5.0, 6.0], "dtype": "float32"}], "e": 32.0}, {"a": [{"__nd__": [1.0, -1.0, 1.0, -1.0], "dtype": "float32"}, {"__nd__": [1.0, 1.0, 1.0, 1.0], "dtype": "float32"}], "e": 0.0}, {"a": [{"__nd__": [], "dtype": "float32"}, {"__nd__": [], "dtype": "float32"}], "e": 0.0}, {"a": [{"__nd__": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], "dtype": "float32"}, {"__nd__": [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5], "dtype": "float32"}], "e": 256.0}, {"a": [{"__nd__": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], "dtype": "float32"}, {"__nd__": [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0, 48.0, 49.0, 50.0, 51.0, 52.0, 53.0, 54.0, 55.0, 56.0, 57.0, 58.0, 59.0, 60.0, 61.0, 62.0, 63.0, 64.0, 65.0, 66.0, 67.0, 68.0, 69.0, 70.0, 71.0, 72.0, 73.0, 74.0, 75.0, 76.0, 77.0, 78.0, 79.0, 80.0, 81.0, 82.0, 83.0, 84.0, 85.0, 86.0, 87.0, 88.0, 89.0, 90.0, 91.0, 92.0, 93.0, 94.0, 95.0, 96.0, 97.0, 98.0, 99.0, 100.0, 101.0, 102.0, 103.0, 104.0, 105.0, 106.0, 107.0, 108.0, 109.0, 110.0, 111.0, 112.0, 113.0, 114.0, 115.0, 116.0, 117.0, 118.0, 119.0, 120.0, 121.0, 122.0, 123.0, 124.0, 125.0, 126.0, 127.0, 128.0, 129.0, 130.0, 131.0, 132.0, 133.0, 134.0, 135.0, 136.0, 137.0, 138.0, 139.0, 140.0, 141.0, 142.0, 143.0, 144.0, 145.0, 146.0, 147.0, 148.0, 149.0, 150.0, 151.0, 152.0, 153.0, 154.0, 155.0, 156.0, 157.0, 158.0, 159.0, 160.0, 161.0, 162.0, 163.0, 164.0, 165.0, 166.0, 167.0, 168.0, 169.0, 170.0, 171.0, 172.0, 173.0, 174.0, 175.0, 176.0, 177.0, 178.0, 179.0, 180.0, 181.0, 182.0, 183.0, 184.0, 185.0, 186.0, 187.0, 188.0, 189.0, 190.0, 191.0, 192.0, 193.0, 194.0, 195.0, 196.0, 197.0, 198.0, 199.0, 200.0, 201.0, 202.0, 203.0, 204.0, 205.0, 206.0, 207.0, 208.0, 209.0, 210.0, 211.0, 212.0, 213.0, 214.0, 215.0, 216.0, 217.0, 218.0, 219.0, 220.0, 221.0, 222.0, 223.0, 224.0, 225.0, 226.0, 227.0, 228.0, 229.0, 230.0, 231.0, 232.0, 233.0, 234.0, 235.0, 236.0, 237.0, 238.0, 239.0, 240.0, 241.0, 242.0, 243.0, 244.0, 245.0, 246.0, 247.0, 248.0, 249.0, 250.0, 251.0, 252.0, 253.0, 254.0, 255.0, 256.0, 257.0, 258.0, 259.0, 260.0, 261.0, 262.0, 263.0, 264.0, 265.0, 266.0, 267.0, 268.0, 269.0, 270.0, 271.0, 272.0, 273.0, 274.0, 275.0, 276.0, 277.0, 278.0, 279.0, 280.0, 281.0, 282.0, 283.0, 284.0, 285.0, 286.0, 287.0, 288.0, 289.0, 290.0, 291.0, 292.0, 293.0, 294.0, 295.0, 296.0, 297.0, 298.0, 299.0, 300.0, 301.0, 302.0, 303.0, 304.0, 305.0, 306.0, 307.0, 308.0, 309.0, 310.0, 311.0, 312.0, 313.0, 314.0, 315.0, 316.0, 317.0, 318.0, 319.0, 320.0, 321.0, 322.0, 323.0, 324.0, 325.0, 326.0, 327.0, 328.0, 329.0, 330.0, 331.0, 332.0, 333.0, 334.0, 335.0, 336.0, 337.0, 338.0, 339.0, 340.0, 341.0, 342.0, 343.0, 344.0, 345.0, 346.0, 347.0, 348.0, 349.0, 350.0, 351.0, 352.0, 353.0, 354.0, 355.0, 356.0, 357.0, 358.0, 359.0, 360.0, 361.0, 362.0, 363.0, 364.0, 365.0, 366.0, 367.0, 368.0, 369.0, 370.0, 371.0, 372.0, 373.0, 374.0, 375.0, 376.0, 377.0, 378.0, 379.0, 380.0, 381.0, 382.0, 383.0, 384.0, 385.0, 386.0, 387.0, 388.0, 389.0, 390.0, 391.0, 392.0, 393.0, 394.0, 395.0, 396.0, 397.0, 398.0, 399.0, 400.0, 401.0, 402.0, 403.0, 404.0, 405.0, 406.0, 407.0, 408.0, 409.0, 410.0, 411.0, 412.0, 413.0, 414.0, 415.0, 416.0, 417.0, 418.0, 419.0, 420.0, 421.0, 422.0, 423.0, 424.0, 425.0, 426.0, 427.0, 428.0, 429.0, 430.0, 431.0, 432.0, 433.0, 434.0, 435.0, 436.0, 437.0, 438.0, 439.0, 440.0, 441.0, 442.0, 443.0, 444.0, 445.0, 446.0, 447.0, 448.0, 449.0, 450.0, 451.0, 452.0, 453.0, 454.0, 455.0, 456.0, 457.0, 458.0, 459.0, 460.0, 461.0, 462.0, 463.0, 464.0, 465.0, 466.0, 467.0, 468.0, 469.0, 470.0, 471.0, 472.0, 473.0, 474.0, 475.0, 476.0, 477.0, 478.0, 479.0, 480.0, 481.0, 482.0, 483.0, 484.0, 485.0, 486.0, 487.0, 488.0, 489.0, 490.0, 491.0, 492.0, 493.0, 494.0, 495.0, 496.0, 497.0, 498.0, 499.0, 500.0, 501.0, 502.0, 503.0, 504.0, 505.0, 506.0, 507.0, 508.0, 509.0, 510.0, 511.0, 512.0, 513.0, 514.0, 515.0, 516.0, 517.0, 518.0, 519.0, 520.0, 521.0, 522.0, 523.0, 524.0, 525.0, 526.0, 527.0, 528.0, 529.0, 530.0, 531.0, 532.0, 533.0, 534.0, 535.0, 536.0, 537.0, 538.0, 539.0, 540.0, 541.0, 542.0, 543.0, 544.0, 545.0, 546.0, 547.0, 548.0, 549.0, 550.0, 551.0, 552.0, 553.0, 554.0, 555.0, 556.0, 557.0, 558.0, 559.0, 560.0, 561.0, 562.0, 563.0, 564.0, 565.0, 566.0, 567.0, 568.0, 569.0, 570.0, 571.0, 572.0, 573.0, 574.0, 575.0, 576.0, 577.0, 578.0, 579.0, 580.0, 581.0, 582.0, 583.0, 584.0, 585.0, 586.0, 587.0, 588.0, 589.0, 590.0, 591.0, 592.0, 593.0, 594.0, 595.0, 596.0, 597.0, 598.0, 599.0, 600.0, 601.0, 602.0, 603.0, 604.0, 605.0, 606.0, 607.0, 608.0, 609.0, 610.0, 611.0, 612.0, 613.0, 614.0, 615.0, 616.0, 617.0, 618.0, 619.0, 620.0, 621.0, 622.0, 623.0, 624.0, 625.0, 626.0, 627.0, 628.0, 629.0, 630.0, 631.0, 632.0, 633.0, 634.0, 635.0, 636.0, 637.0, 638.0, 639.0, 640.0, 641.0, 642.0, 643.0, 644.0, 645.0, 646.0, 647.0, 648.0, 649.0, 650.0, 651.0, 652.0, 653.0, 654.0, 655.0, 656.0, 657.0, 658.0, 659.0, 660.0, 661.0, 662.0, 663.0, 664.0, 665.0, 666.0, 667.0, 668.0, 669.0, 670.0, 671.0, 672.0, 673.0, 674.0, 675.0, 676.0, 677.0, 678.0, 679.0, 680.0, 681.0, 682.0, 683.0, 684.0, 685.0, 686.0, 687.0, 688.0, 689.0, 690.0, 691.0, 692.0, 693.0, 694.0, 695.0, 696.0, 697.0, 698.0, 699.0, 700.0, 701.0, 702.0, 703.0, 704.0, 705.0, 706.0, 707.0, 708.0, 709.0, 710.0, 711.0, 712.0, 713.0, 714.0, 715.0, 716.0, 717.0, 718.0, 719.0, 720.0, 721.0, 722.0, 723.0, 724.0, 725.0, 726.0, 727.0, 728.0, 729.0, 730.0, 731.0, 732.0, 733.0, 734.0, 735.0, 736.0, 737.0, 738.0, 739.0, 740.0, 741.0, 742.0, 743.0, 744.0, 745.0, 746.0, 747.0, 748.0, 749.0, 750.0, 751.0, 752.0, 753.0, 754.0, 755.0, 756.0, 757.0, 758.0, 759.0, 760.0, 761.0, 762.0, 763.0, 764.0, 765.0, 766.0, 767.0, 768.0, 769.0, 770.0, 771.0, 772.0, 773.0, 774.0, 775.0, 776.0, 777.0, 778.0, 779.0, 780.0, 781.0, 782.0, 783.0, 784.0, 785.0, 786.0, 787.0, 788.0, 789.0, 790.0, 791.0, 792.0, 793.0, 794.0, 795.0, 796.0, 797.0, 798.0, 799.0, 800.0, 801.0, 802.0, 803.0, 804.0, 805.0, 806.0, 807.0, 808.0, 809.0, 810.0, 811.0, 812.0, 813.0, 814.0, 815.0, 816.0, 817.0, 818.0, 819.0, 820.0, 821.0, 822.0, 823.0, 824.0, 825.0, 826.0, 827.0, 828.0, 829.0, 830.0, 831.0, 832.0, 833.0, 834.0, 835.0, 836.0, 837.0, 838.0, 839.0, 840.0, 841.0, 842.0, 843.0, 844.0, 845.0, 846.0, 847.0, 848.0, 849.0, 850.0, 851.0, 852.0, 853.0, 854.0, 855.0, 856.0, 857.0, 858.0, 859.0, 860.0, 861.0, 862.0, 863.0, 864.0, 865.0, 866.0, 867.0, 868.0, 869.0, 870.0, 871.0, 872.0, 873.0, 874.0, 875.0, 876.0, 877.0, 878.0, 879.0, 880.0, 881.0, 882.0, 883.0, 884.0, 885.0, 886.0, 887.0, 888.0, 889.0, 890.0, 891.0, 892.0, 893.0, 894.0, 895.0, 896.0, 897.0, 898.0, 899.0, 900.0, 901.0, 902.0, 903.0, 904.0, 905.0, 906.0, 907.0, 908.0, 909.0, 910.0, 911.0, 912.0, 913.0, 914.0, 915.0, 916.0, 917.0, 918.0, 919.0, 920.0, 921.0, 922.0, 923.0, 924.0, 925.0, 926.0, 927.0, 928.0, 929.0, 930.0, 931.0, 932.0, 933.0, 934.0, 935.0, 936.0, 937.0, 938.0, 939.0, 940.0, 941.0, 942.0, 943.0, 944.0, 945.0, 946.0, 947.0, 948.0, 949.0, 950.0, 951.0, 952.0, 953.0, 954.0, 955.0, 956.0, 957.0, 958.0, 959.0, 960.0, 961.0, 962.0, 963.0, 964.0, 965.0, 966.0, 967.0, 968.0, 969.0, 970.0, 971.0, 972.0, 973.0, 974.0, 975.0, 976.0, 977.0, 978.0, 979.0, 980.0, 981.0, 982.0, 983.0, 984.0, 985.0, 986.0, 987.0, 988.0, 989.0, 990.0, 991.0, 992.0, 993.0, 994.0, 995.0, 996.0, 997.0, 998.0, 999.0], "dtype": "float32"}], "e": 499500.0}, {"a": [{"__nd__": [-1.4308730363845825, -0.9365476965904236, 0.3939383625984192, -0.5240866541862488, 0.5256162285804749, 0.8073236346244812, -1.4435313940048218, 1.0170637369155884, -0.5956486463546753, 2.0940849781036377, 0.7286889553070068, -0.5389894247055054, 0.20148779451847076, 0.2149779200553894, 0.3295181095600128, 1.3932604789733887, 0.5328440070152283, 1.6606374979019165, 0.6342201232910156, -0.17774108052253723, -1.5244078636169434, -1.560539722442627, 0.204787477850914, 0.6909785866737366, 1.4433056116104126, -0.613301694393158, 0.07777395099401474, -0.8129783272743225, -0.42233335971832275, -2.2794487476348877, 0.7388424277305603, -0.7226053476333618, -0.07269526273012161, 0.445732980966568, 0.33908846974372864, -1.3142318725585938, 1.5181677341461182, 0.7573140263557434, 0.5001350045204163, -0.6595975160598755, -1.6042755842208862, -1.6301579475402832, -0.20841862261295319, 0.1606714129447937, -0.7033604979515076, 1.8294451236724854, 0.3011413514614105, 0.9985196590423584, 0.9065691232681274, -1.0704716444015503, 0.9938796162605286, 0.45020246505737305, -1.268629789352417, 0.7234396934509277, 1.0719754695892334, 0.9765949249267578, 1.0518536567687988, -1.4101163148880005, -0.24749669432640076, -0.545717179775238, 0.054290756583213806, 0.10624998807907104, 1.1260778903961182, 0.7521507740020752, -0.8289487361907959, 0.9014636278152466, 1.6261274814605713, 0.3063559830188751, -1.5056171417236328, -0.6447046399116516, -0.1795082688331604, -1.6233521699905396, -0.13829556107521057, -1.2726643085479736, 0.7926174402236938, 0.6843770742416382, -0.07646868377923965, -1.2547872066497803, 0.28921037912368774, -0.6374709606170654, -0.6877455115318298, 1.1100332736968994, -0.7571494579315186, -1.376811146736145, 0.7903707027435303, 1.7780290842056274, 1.609283208847046, 0.781023383140564, 2.092095375061035, -0.35552698373794556, 2.175355911254883, 2.0984630584716797, -0.34232473373413086, 0.4131457805633545, -1.2868221998214722, -1.0262171030044556, 0.5658305883407593, -1.163601279258728, 0.2148730754852295, 0.46271616220474243, 0.14006251096725464, 0.978299617767334, 1.9572845697402954, 0.1955515742301941, 1.022183895111084, 1.5142062902450562, -0.43712949752807617, -0.6492400169372559, -2.4134414196014404, 0.45204973220825195, 1.076810359954834, 0.46086743474006653, 2.1272342205047607, 1.557761549949646, 0.8454576134681702, -0.008923848159611225, -0.8509684205055237, 0.13937361538410187, 0.5317212343215942, -0.03649395704269409, -0.275094211101532, -0.35509440302848816, 0.7307364344596863, -1.680033564567566, -0.288077712059021, 0.10188166797161102, 0.7264209389686584, 0.733652651309967, -0.19755499064922333, -0.5546512007713318, -1.1421971321105957, 1.1985130310058594, 0.617045521736145, 0.23711633682250977, -0.6961370706558228, -1.097367763519287, 1.201295256614685, 0.020417068153619766, 0.505156397819519, 1.8359317779541016, -1.5018198490142822, -0.7438716292381287, 0.4010504186153412, 0.658054530620575, -1.0111340284347534, -0.04821792617440224, -0.5738697648048401, -0.5485254526138306, 0.4167379140853882, 0.22900953888893127, 1.2110986709594727, 0.6385900974273682, 1.5287163257598877, -0.7903888821601868, 0.4405798614025116, -1.400289535522461, -1.6474456787109375, -1.3805242776870728, 0.7277050614356995, 0.212717667222023, -1.6312605142593384, 1.317460298538208, 0.9747344255447388, -0.18016573786735535, -0.24211181700229645, -0.011861858889460564, -1.2234693765640259, 2.3512144088745117, -0.4705912470817566, -0.19651833176612854, -0.32857638597488403, -0.6737297773361206, -0.9136252403259277, 0.331112265586853, -1.2441186904907227, -1.0527797937393188, -0.8169077634811401, 0.8004215359687805, -1.0048894882202148, -1.0055924654006958, 0.31929150223731995, -1.14422607421875, -0.33579885959625244, 0.8835270404815674, 0.40319153666496277, 0.021310899406671524, 1.5186179876327515, -0.42978399991989136, -0.6199194192886353, -1.771889090538025, 0.021381322294473648, 0.6204207539558411, -0.6825700998306274, 1.8733830451965332, 1.0091933012008667, 0.9130192995071411, -1.2213808298110962, 0.2806476056575775, 0.354871928691864, -2.0599777698516846, -0.03765140101313591, -0.25556859374046326, -1.7551651000976562, 0.4793957471847534, 2.041800022125244, 0.738923192024231, -0.8675675988197327, -0.27166667580604553, 1.112744688987732, -0.8480567336082458, 0.34445542097091675, 0.022162960842251778, 0.17062419652938843, 1.6503689289093018, -1.5371965169906616, 1.1659936904907227, 0.2589932382106781, -0.10007935017347336, -1.6778497695922852, 0.30204808712005615, -0.417751282453537, -2.9574778079986572, -1.3845393657684326, -1.2132296562194824, -1.0257529020309448, -0.624639093875885, -0.8458772897720337, 1.555108666419983, 1.0633399486541748, -0.08986803144216537, 1.3869245052337646, 2.1368608474731445, 0.4853801131248474, -1.3313533067703247, 0.37197986245155334, 1.4307262897491455, -1.0069193840026855, -0.011692635715007782, 0.9369845390319824, 1.1858190298080444, 0.8511761426925659, 0.372438907623291, -0.1425822228193283, -0.5730961561203003, 1.004692792892456, 0.927515983581543, -0.48820918798446655, -0.694405198097229, 1.2596943378448486, 1.0016613006591797, -0.7710831165313721, -1.6599968671798706, 0.11470160633325577, 0.8080424070358276, 0.9613726735115051, 0.8157538771629333, -1.5544813871383667, 0.0425310917198658, -0.38969531655311584, -0.744293212890625, 2.635220527648926, -0.005728777032345533, 0.24792888760566711, -0.3079701066017151, 0.680010974407196, 1.1633415222167969, -0.28989240527153015, -0.03596894443035126, -0.4548233151435852, -1.0324136018753052, -0.19148090481758118, 0.3409649133682251, 0.01000966876745224, -0.25428634881973267, -1.8895297050476074, 0.32532215118408203, 0.1557835340499878, 1.5238882303237915, -0.6600499749183655, -1.4048564434051514, -1.955318570137024, -0.6700511574745178, -0.09457703679800034, 0.227790966629982, -1.6115894317626953, -0.5926144123077393, 0.9875521063804626, 0.03705676272511482, -0.36717021465301514, -0.3957708179950714, -1.6085106134414673, -1.4339993000030518, -0.5545377731323242, 1.198715329170227, 1.1259808540344238, -0.18314887583255768, 0.5977662801742554, 0.3952401578426361, 1.1340073347091675, -0.7283825278282166, 0.24113604426383972, 0.9276218414306641, -1.284781813621521, -0.09259328246116638, 0.23021844029426575, 0.3994883894920349, 0.5036541223526001, 0.5753886699676514, 0.9882739186286926, 1.3803998231887817, 0.3658527135848999, 2.1899876594543457, -0.4826064109802246, -0.19987428188323975, 0.32121720910072327, -0.7842108607292175, -1.4786951541900635, -1.3203150033950806, 1.1606502532958984, -0.6564350724220276, -1.3429402112960815, 1.8864638805389404, 0.3718126714229584, 0.19189928472042084, -0.9652628898620605, 0.26796606183052063, -0.5162485837936401, 0.23687796294689178, -0.035027746111154556, -0.2418321967124939, 0.20116043090820312, -0.9183903932571411, -0.4392107427120209, -0.8089579343795776, 0.37102481722831726, 0.6901586055755615, 0.7854767441749573, -1.9472455978393555, 1.7085165977478027, -0.23751011490821838, -0.1411541998386383, -0.06814690679311752, -0.08034011721611023, 1.234183669090271, 0.10982222110033035, 0.27967944741249084, 0.9303843975067139, 1.2969568967819214, -1.0307250022888184, -1.2348475456237793, -1.9375065565109253, -1.9187953472137451, 0.2924160361289978, 0.07922576367855072, -0.041612427681684494, 0.32617679238319397, 0.2947034239768982, 1.187034249305725, 1.6979924440383911, -0.8050826787948608, 1.4942845106124878, 1.067878007888794, -0.7923129200935364, 0.18996331095695496, -0.41944774985313416, -1.6818803548812866, 0.05319451913237572, -1.6108644008636475, 0.058124471455812454, -1.0262664556503296, -0.31746113300323486, 0.8177950382232666, 0.1312054693698883, -0.7861301898956299, 0.24177315831184387, 0.7019067406654358, 0.9791248440742493, -0.8152059316635132, 1.141300916671753, -0.15711970627307892, -0.5343417525291443, 1.1895697116851807, 0.6387221813201904, 0.6946814656257629, 0.14972127974033356, -1.7747600078582764, 0.47948113083839417, 0.5786226391792297, -0.09525084495544434, 1.7100249528884888, 0.15257413685321808, 1.719977617263794, 1.211655855178833, -0.639645516872406, -0.7929854989051819, -0.27970123291015625, -0.011301700957119465, 0.7271533608436584, -0.5603079199790955, -1.0064510107040405, 2.1801326274871826, 0.0803508386015892, 0.3363751173019409, 1.7954455614089966, 0.16987255215644836, -2.258159875869751, 0.5348727107048035, -0.5618482828140259, 1.0533018112182617, -1.735448956489563, 0.2498522251844406, 0.866096019744873, 1.4782835245132446, 1.035409688949585, 1.0028226375579834, 0.06334531307220459, 0.046755265444517136, -0.930253267288208, 1.0302835702896118, 1.6178770065307617, 0.6780112385749817, -0.7965879440307617, -1.3283442258834839, -0.4452546238899231, 1.6377900838851929, 0.24026094377040863, -0.8940567970275879, -0.2811407744884491, 0.8913431167602539, -0.5787273049354553, -0.3997781276702881, -0.4626433253288269, -0.8151618242263794, 2.007002592086792, -0.7240394353866577, 0.3463089168071747, 1.2521545886993408, -0.033779386430978775, -0.32801416516304016, -0.0765947699546814, 0.07505486160516739, -0.12982900440692902, -0.3633175492286682, -0.30605167150497437, 0.187755286693573, 1.828235149383545, -0.3324918746948242, 0.8723645210266113, -0.10737054795026779, -0.8790488243103027, -1.0642027854919434, 0.4014718234539032, -1.2158823013305664, 0.5147815942764282, -0.2578538954257965, 1.7584397792816162, -0.9524922370910645, 1.1675196886062622, 1.4620329141616821, 0.9030712842941284, -0.4440006613731384, 0.43268299102783203, 0.22345538437366486, -0.06002148240804672, 0.9848201870918274, 0.011709755286574364, 0.330403596162796, 1.167623519897461, -0.012188097462058067, -0.27211901545524597, 0.2897661626338959, -0.4068698585033417, -2.209638833999634, -0.4181366264820099, -0.07933689653873444, -1.080660104751587, -1.7745989561080933, 0.5386615991592407, -0.573017954826355, 0.7187828421592712, 0.5424395203590393, 1.60002601146698, 0.6601814031600952, 0.9939338564872742, 1.4474812746047974, -0.25401368737220764, 1.1912671327590942, 1.2797553539276123, -1.70090913772583, -0.9673371911048889, 0.2333347350358963, 1.7908897399902344, -0.2471054047346115, -1.1931390762329102, -1.4922412633895874, 0.3852660059928894, 0.2829790413379669, -0.5740768909454346, 0.35823971033096313, -0.3057291507720947, 1.1950576305389404, -1.331843376159668, -0.4177767336368561, 1.1652542352676392, -0.5727814435958862, -0.8080522418022156, -0.2404862642288208, 1.8800593614578247, 0.3168690800666809, 1.4478180408477783, -0.8707929849624634, -0.15220879018306732, 0.729386568069458, 0.2892310321331024, -0.48888376355171204, -2.177476406097412, 0.5650877356529236, 0.7822892069816589, 0.5063236951828003, -1.6548436880111694, -1.386454463005066, -0.5620846748352051, -0.9999793767929077, 1.0589004755020142, -1.074431300163269, -0.3167409300804138, -1.0917571783065796, -1.72322416305542, 0.9975363612174988, 0.16533522307872772, -0.6949930191040039, 0.025675581768155098, -2.1625258922576904, 1.5679715871810913, -1.027205467224121, -1.8989440202713013, -1.0529544353485107, 0.2223414182662964, 0.008276881650090218, 0.6637474298477173, 0.6907651424407959, 0.04439130425453186, 1.0215630531311035, -1.8171106576919556, 0.4654806852340698, 0.46425801515579224, 0.34442248940467834, -0.16081835329532623, -0.5316556096076965, -0.6032884120941162, -1.6929256916046143, -0.5057145357131958, 1.0633869171142578, 1.4348303079605103, -0.4834940433502197, -0.5456142425537109, -0.5144839882850647, -0.15219201147556305, -0.008752034045755863, 0.4625498950481415, 1.4119434356689453, -2.021761894226074, -1.0722917318344116, -0.4851507246494293, -0.07308818399906158, 0.3856561481952667, 1.7544974088668823, -0.9338286519050598, 0.6328231692314148, -0.0901947095990181, 0.05776135250926018, 0.3012858033180237, 0.8529576063156128, 0.12562595307826996, -1.0921865701675415, 0.47312209010124207, 0.48740285634994507, -1.5396069288253784, 0.43796244263648987, -0.07315079122781754, -0.16584374010562897, 0.089850053191185, 1.728043794631958, -1.6476784944534302, 0.18464608490467072, -1.8585503101348877, 0.017171038314700127, 0.9135268926620483, -1.8001176118850708, 1.1360020637512207, 0.47682806849479675, -0.11256895214319229, -0.16898353397846222, -0.002513842424377799, 0.07735597342252731, -0.019139299169182777, 1.0685492753982544, 0.8237801194190979, -0.8565617203712463, 1.1133720874786377, 1.8674652576446533, 1.6623773574829102, 0.351529598236084, -1.5072641372680664, -0.4465850591659546, -0.046405989676713943, -0.008271829225122929, -1.3854056596755981, 0.6855357885360718, 0.9826382994651794, -1.2495248317718506, -0.6528032422065735, -0.9024869799613953, 0.14608117938041687, 1.2552098035812378, -1.9551150798797607, 1.6894655227661133, 1.1025241613388062, -1.1692121028900146, 1.7550305128097534, 1.31218683719635, 0.6749345064163208, 0.46759143471717834, -0.7571340203285217, 0.6318350434303284, -0.6803608536720276, 0.20621350407600403, 1.1815780401229858, -0.22688981890678406, -0.4920097589492798, 0.6656067371368408, 1.0963557958602905, 1.1283787488937378, -0.0762191042304039, 1.528476595878601, 1.7857615947723389, -0.1891838163137436, 0.7387951016426086, -1.6403172016143799, -0.01822107657790184, -0.9122133851051331, 0.26354092359542847, -0.49091824889183044, -2.212907552719116, 0.7390038371086121, -0.8906635642051697, 1.3007785081863403, 1.5687825679779053, 0.7046210765838623, 0.03840167075395584, 0.3563295006752014, -1.3063056468963623, 0.2040363848209381, 1.082484245300293, 0.8197933435440063, -0.9222872853279114, 1.6289308071136475, 0.6234108805656433, -0.8351908326148987, 0.5622901320457458, -0.8299374580383301, -0.37776607275009155, 0.1487773209810257, 1.9583529233932495, 2.59291934967041, -1.4061908721923828, 0.032713454216718674, -0.06492731720209122, 0.7435052990913391, 0.448363333940506, -0.523817241191864, 0.3940980136394501, -0.574977457523346, -0.7496899962425232, 0.6257613301277161, 0.24604007601737976, -0.5346199870109558, 0.7093866467475891, -0.33913224935531616, -0.7180063128471375, -0.06796730309724808, 0.6647339463233948, -0.5609588623046875, 1.48878014087677, -0.1871170997619629, 0.3405504524707794, 0.8316214084625244, 0.047234661877155304, -0.5871152877807617, -0.15932206809520721, 0.9937552809715271, 1.2939176559448242, 0.6753493547439575, -0.40718165040016174, -0.9117029905319214, -0.5167427659034729, -0.4108271896839142, 0.27849289774894714, 0.03003385290503502, -0.7178539037704468, 0.7460328340530396, -0.6651095747947693, 0.13022887706756592, -0.6874650120735168, 1.091620683670044, 0.4599091112613678, 0.25614944100379944, 1.1476514339447021, 1.0012176036834717, -1.2107040882110596, 0.2608497440814972, -0.9630918502807617, -0.5248177647590637, -0.36728620529174805, -1.3648931980133057, -1.1657456159591675, 0.14103932678699493, 0.1521439254283905, 0.5548079609870911, -0.1032685935497284, -0.35565486550331116, -0.2847822904586792, -0.2787333130836487, -0.7378696203231812, 0.11757475882768631, -0.9133217930793762, -0.9965865015983582, -1.4811229705810547, 0.11018602550029755, -0.6905060410499573, -0.26156187057495117, 0.07032160460948944, -0.9045930504798889, 0.16548728942871094, 0.025294043123722076, -0.052687935531139374, -1.8807169198989868, 0.9775583148002625, 0.48525556921958923, 0.5572009682655334, 1.660404086112976, -1.0258383750915527, 1.8765124082565308, -0.21584850549697876, 0.8741844296455383, 1.1510072946548462, -0.9928708672523499, 0.28270047903060913, -0.5555216073989868, 0.4421899616718292, -1.4087910652160645, 1.6686869859695435, 0.4315968453884125, -0.22214967012405396, -0.8272117376327515, 1.3190793991088867, -0.48535045981407166, 0.7848042845726013, -1.1435171365737915, 0.5414851903915405, -1.8255667686462402, 0.34532463550567627, -0.9641937613487244, -0.996963620185852, 0.8987588882446289, 0.6652022004127502, -0.6141889095306396, -0.7536534070968628, 1.026781439781189, -1.508522629737854, -0.3060855567455292, -0.7399300932884216, -0.025164812803268433, -0.0981929674744606, -0.2049802839756012, -1.2251510620117188, 0.20734046399593353], "dtype": "float32"}, {"__nd__": [0.42857539653778076, -0.4615797996520996, 1.6358219385147095, -0.474618136882782, -0.034504443407058716, 1.8831857442855835, -0.31297481060028076, -0.1476706564426422, -1.4787448644638062, -0.022853419184684753, -0.4505423903465271, 0.25981569290161133, -1.086639404296875, 1.1286331415176392, 0.4736552834510803, 0.830906867980957, -0.39225319027900696, -1.7512526512145996, 0.29709139466285706, -0.05843622237443924, -0.5086452960968018, 1.551846981048584, 0.2485225647687912, 0.10914047062397003, 0.4158744215965271, 0.31523630023002625, 0.38247910141944885, 0.7527256608009338, 0.09407775849103928, -0.4439862370491028, -0.8077064156532288, 0.20365816354751587, 0.45998120307922363, 0.751063346862793, -0.7941651344299316, -1.4778504371643066, -1.4160246849060059, 1.4076536893844604, 0.5706514716148376, -0.03077283874154091, -0.30899176001548767, 2.1146576404571533, 0.7628293633460999, 0.6769125461578369, -0.5292275547981262, 1.120744228363037, -0.39527156949043274, -0.7570596933364868, 1.3240522146224976, 0.3322330117225647, 0.6638832092285156, -1.4511802196502686, -0.3807954490184784, -0.8518815040588379, 0.1863868236541748, -0.12990990281105042, -0.05794907733798027, -0.783068060874939, 1.3190326690673828, -1.3165292739868164, 0.3920709490776062, -0.10133402794599533, 0.9415317177772522, -1.0091716051101685, 0.9322398900985718, -0.8812217712402344, 0.1722477674484253, -0.2675314247608185, 2.2490768432617188, -1.3492934703826904, -0.736149787902832, -1.88111412525177, -1.1296064853668213, -0.308558851480484, -0.49849385023117065, -0.956212043762207, -0.8087408542633057, -0.3629549741744995, -0.04016965627670288, 1.4649516344070435, 0.2352965921163559, -0.19121336936950684, -0.5027742385864258, 0.5071925520896912, 1.1359604597091675, -0.4114369750022888, 0.7880625128746033, 0.35879483819007874, -2.21199107170105, 1.745864748954773, 0.7736172676086426, -1.4906491041183472, -2.0347020626068115, 0.06497063487768173, -0.6170598268508911, 0.37728574872016907, -0.5020468235015869, 1.0970245599746704, -0.6646296381950378, 0.3134814202785492, 0.5576236248016357, -0.36914804577827454, 0.7932361960411072, 0.03983322158455849, -0.08145835995674133, -0.2820911407470703, -2.408773422241211, -1.0699982643127441, 1.7353843450546265, -0.43728941679000854, -0.8461863994598389, -0.256561279296875, -0.2751617431640625, 1.9864327907562256, -1.1484371423721313, -0.7563949823379517, 0.5706906914710999, -1.8025970458984375, -1.6421940326690674, -1.3029375076293945, -0.7441561222076416, 0.9496093988418579, 1.2524614334106445, 0.44803428649902344, 1.8891043663024902, -1.8980625867843628, 0.9923547506332397, -0.7252270579338074, 0.43406081199645996, -0.3137722909450531, -1.449960708618164, 1.0977739095687866, 1.3811671733856201, 0.22376267611980438, -0.8387342095375061, 1.9746509790420532, -2.3347342014312744, 0.18114668130874634, -0.8944669365882874, -0.7169233560562134, 0.9430264830589294, -0.4633537828922272, 0.9701089859008789, 0.23444508016109467, 1.3645023107528687, 0.9334065318107605, 0.8338948488235474, 0.02488575130701065, -0.0587533637881279, -0.9608606696128845, 0.43229714035987854, 0.707412838935852, -0.6826208233833313, 0.37281548976898193, -0.7692813277244568, -0.04288169741630554, -1.074188470840454, 0.000835770508274436, 1.0466481447219849, 0.2667534351348877, -1.724600911140442, 0.6398450136184692, -0.33942434191703796, 1.2468968629837036, 0.6780455112457275, 0.6622877717018127, -0.30937114357948303, 0.1354466676712036, -0.7553353309631348, 0.07295462489128113, 0.754069983959198, -1.4787640571594238, -0.3378143906593323, 0.7111064195632935, -1.0236634016036987, -0.16343259811401367, 0.46479088068008423, 0.30839449167251587, 0.5818449854850769, 0.7438522577285767, 1.171810507774353, -0.31111404299736023, 0.47469261288642883, 0.3913843035697937, -1.1664668321609497, -0.4078488051891327, 0.7073877453804016, 0.3410913050174713, 0.6739072799682617, -0.019496791064739227, -0.039551205933094025, 0.9257250428199768, -1.9142783880233765, -1.6402393579483032, 0.03392254188656807, 0.24936829507350922, 0.6408901214599609, 0.6469996571540833, 0.17896638810634613, -1.147377371788025, 0.49732351303100586, 0.19050036370754242, -0.9869709014892578, -1.6634469032287598, -0.4550454318523407, 0.6403217315673828, -0.0019724539015442133, 1.1045401096343994, -0.3841361999511719, -0.7191605567932129, -0.6053327322006226, 0.02114175818860531, 0.24608264863491058, 0.8584392070770264, -0.1378541886806488, -1.17497718334198, -0.8070960640907288, 0.21942690014839172, -0.15676113963127136, 0.13890771567821503, -0.592479407787323, -0.4391149878501892, -0.5024964809417725, -0.6190703511238098, 0.1322982758283615, 1.5914846658706665, -0.02212558314204216, 0.11661865562200546, -1.866576075553894, 0.47534802556037903, -0.26738524436950684, 1.260846495628357, 0.4861011207103729, 1.3668718338012695, -0.948197066783905, 0.6733757257461548, 0.8689442873001099, -0.3623148202896118, 1.1539888381958008, -1.3145896196365356, 0.2534019351005554, -0.41982516646385193, 0.8102177977561951, -1.8861109018325806, 0.8538796305656433, 1.6386361122131348, -0.9803637266159058, 0.9242311120033264, -0.8197365999221802, 0.8952534198760986, 1.1385596990585327, -1.1020464897155762, 1.6515023708343506, -0.05442626774311066, 0.0954127088189125, 0.8064193725585938, 0.9714723229408264, 0.10905805230140686, 0.6925946474075317, 0.7065527439117432, -0.9963404536247253, 0.34579160809516907, 0.28537341952323914, 0.10330110788345337, 0.7528843879699707, -0.4232139587402344, -1.638893961906433, 0.6890044212341309, 0.4517959952354431, 0.07076902687549591, -0.4028244614601135, 2.220086097717285, 0.17099419236183167, -0.9339383840560913, 0.372504860162735, 0.5719327926635742, -1.2767531871795654, 0.6995757818222046, -1.9688208103179932, -0.3365590572357178, 0.903825044631958, 1.1429216861724854, 0.4324578046798706, 0.8328549861907959, -0.7388101816177368, 0.14480692148208618, 0.7200821042060852, -1.6201797723770142, 0.34329527616500854, 0.675117552280426, 1.1447750329971313, 0.8269855976104736, 1.055091381072998, 0.8943554759025574, -0.8038892149925232, -0.8221579194068909, -0.5063837170600891, -1.1462398767471313, 0.8566256761550903, 1.717615008354187, 1.3391257524490356, 0.27973222732543945, 1.0969526767730713, 0.1197744682431221, 0.5295883417129517, 1.1670949459075928, -0.9499077796936035, -0.5178133249282837, 0.46600452065467834, 2.921067953109741, 0.6862629652023315, 0.8346430063247681, 1.4624568223953247, -0.4018893539905548, -1.9971792697906494, -1.0548195838928223, -0.9343591928482056, 1.7564266920089722, -0.21062423288822174, -0.21386542916297913, -0.20889008045196533, -0.24453406035900116, -0.08424164354801178, 0.04181750863790512, 1.113952875137329, 0.15272468328475952, -1.4444342851638794, 0.7407429814338684, 0.43733319640159607, -0.40616583824157715, 0.17139701545238495, -0.6629320383071899, 2.1450564861297607, 1.049414873123169, 0.24223124980926514, -0.8446800112724304, 0.5901039838790894, 0.5682567954063416, 0.02958524413406849, -0.2944616377353668, 1.9179577827453613, -0.3761037588119507, 0.8806548118591309, 1.1802974939346313, -1.0604578256607056, 0.20894409716129303, 0.9720023274421692, 0.05748056247830391, -1.419209599494934, 0.6040691137313843, 1.976104974746704, 0.4773384928703308, 0.33024919033050537, -0.7147078514099121, -0.1590028554201126, 0.4649312496185303, -1.2616314888000488, -0.8769253492355347, 0.0707966610789299, -0.0051007578149437904, 0.0958719402551651, -0.04798347130417824, 1.3570218086242676, -0.10265976935625076, -0.04209119826555252, -0.7761996984481812, -0.7988973259925842, -0.10113224387168884, 2.4559030532836914, 1.0870397090911865, -1.2390189170837402, 1.541062355041504, 0.02587065100669861, -1.2065823078155518, 0.28739944100379944, -0.9152636528015137, -0.6248505711555481, -0.92230224609375, -0.1375892013311386, -1.9283685684204102, 0.942462146282196, 1.6167430877685547, -0.14456705749034882, 0.07195775955915451, 0.4331103265285492, -0.1901324987411499, -0.22411583364009857, 0.6840242147445679, 1.0151737928390503, -0.353082537651062, 0.6874746680259705, 0.3333398699760437, -0.5548705458641052, -0.03623038902878761, -1.7959933280944824, -1.0657625198364258, 0.2939242124557495, 0.5550923347473145, 1.033398151397705, 1.4243401288986206, -1.0447086095809937, 0.3021436035633087, -1.5511218309402466, -0.06074756011366844, -0.5504490733146667, 0.42206889390945435, -0.11526458710432053, 0.04253500699996948, -0.5438191890716553, -0.012388291768729687, 1.2584534883499146, 0.7541212439537048, 0.03322547674179077, -1.5662028789520264, -1.686062216758728, -0.04560980573296547, 0.33955371379852295, 0.5570898652076721, 0.23113416135311127, -0.7263380289077759, -1.625759482383728, 0.7857317924499512, 0.5438005924224854, 1.145477294921875, -1.570400595664978, 0.6059824824333191, 0.6625470519065857, -1.0263681411743164, -0.9400079846382141, -1.0009769201278687, -0.7807016968727112, -0.11021098494529724, -0.3808935582637787, 1.26681649684906, 0.3789503872394562, -0.5834236741065979, 0.2057897299528122, -0.693235456943512, -2.1293253898620605, 0.2951065003871918, -0.547156810760498, 1.1826531887054443, -0.09893205761909485, 1.3153431415557861, 0.7906134128570557, -1.4501395225524902, 1.6005592346191406, -0.3908696174621582, -0.046649739146232605, -1.077283263206482, -0.5385724902153015, 2.524547815322876, 0.1582302302122116, -3.4155445098876953, 1.6961214542388916, 0.20524729788303375, -0.6006725430488586, 0.63702392578125, 0.36031925678253174, -0.6385360956192017, 1.492902398109436, 0.5451229810714722, 1.0904427766799927, 0.21687771379947662, -0.029915571212768555, 0.17139403522014618, -0.9469203352928162, -0.8301042318344116, 0.8584113717079163, -0.08894739300012589, 0.5595929622650146, -0.014358907006680965, -0.44800660014152527, 1.0207141637802124, -0.5634739995002747, -0.9139635562896729, -0.026693278923630714, 1.0024940967559814, 0.9874118566513062, -1.0747066736221313, -1.0290405750274658, -0.5522263646125793, -1.014845848083496, -0.06132528558373451, 0.7887114882469177, -0.35625556111335754, -1.7597259283065796, -1.8265544176101685, -1.6474740505218506, -0.536612331867218, 0.3807518184185028, -1.643375039100647, 0.2670217454433441, 0.526409387588501, 2.3122572898864746, -1.1590049266815186, -0.0003210444119758904, 0.6173681616783142, 0.3569921553134918, 0.7740841507911682, 0.32246217131614685, 0.6902486681938171, -0.18060623109340668, -0.12132468074560165, -1.7158794403076172, 1.9300044775009155, -1.3649934530258179, 0.10025154799222946, 0.12258420884609222, -0.8555418252944946, -1.5986638069152832, 0.21461743116378784, -1.6666525602340698, 1.0290800333023071, 2.6229288578033447, 1.3347156047821045, -2.567469835281372, 0.4103836417198181, 0.06549365073442459, -0.19583430886268616, 0.8559922575950623, -0.4732327163219452, 0.5428506731987, -0.6504025459289551, -0.9958919882774353, 0.5813285112380981, 1.0625628232955933, -0.5651317238807678, 1.5843234062194824, -0.15029369294643402, -0.8666984438896179, -0.7040934562683105, -0.5942860245704651, 0.7195042371749878, 0.7611873149871826, 0.10544157028198242, 1.3723598718643188, -0.8672569990158081, 0.47527265548706055, 0.3142441213130951, -2.093693494796753, 0.3544659912586212, -0.43269622325897217, 1.548298716545105, -0.42225348949432373, 1.3777433633804321, 0.563422679901123, -1.4222267866134644, 1.17324697971344, 0.7339150309562683, 0.4170379638671875, -0.09591604769229889, 1.9135128259658813, -0.20042972266674042, 1.2296861410140991, 0.697350800037384, -0.06591953337192535, -0.6278607845306396, 0.4535508453845978, 0.03590227663516998, -1.053547739982605, 0.50337815284729, 0.6581153869628906, 0.18528778851032257, -1.5995413064956665, 0.6869213581085205, -0.19214920699596405, 0.9303918480873108, -0.2254568487405777, -0.5426971912384033, -0.048241399228572845, 0.00965255219489336, 2.2067525386810303, -0.9256017208099365, -1.4133273363113403, -0.09910517930984497, -0.018230635672807693, -1.7650883197784424, -1.6369675397872925, -0.8604192733764648, -0.24710816144943237, -0.297048419713974, 0.8219020962715149, -0.03225337341427803, -0.7554942965507507, -0.08738335967063904, 0.27332592010498047, 0.5792098045349121, -2.8231258392333984, 0.027893228456377983, 0.28044408559799194, 1.4017598628997803, -0.369850754737854, 0.11101135611534119, -1.604185700416565, 0.7679247856140137, 0.978070855140686, -0.8814226984977722, 0.07550026476383209, -0.4859553575515747, 1.0375175476074219, -0.2597729563713074, 0.17390337586402893, 0.21562284231185913, -0.3543466329574585, 0.20269016921520233, 1.6026043891906738, -0.0060569895431399345, -0.7423028349876404, 1.154821515083313, 1.8885499238967896, 0.951188325881958, 0.05783172324299812, -0.005506924353539944, 0.6501300930976868, 1.0480685234069824, 2.881962776184082, 2.376699686050415, -1.004239797592163, 0.8267138600349426, 1.4659371376037598, 0.7168585062026978, -0.03894391655921936, -0.6176095604896545, 0.6655254364013672, 0.7321279644966125, 0.3740454614162445, -0.8564171195030212, -1.8740031719207764, 1.6994582414627075, -0.9414341449737549, -0.14612087607383728, -1.3343255519866943, 0.9651073217391968, -0.4436829388141632, 0.0951487272977829, 2.5381147861480713, 1.0653390884399414, 0.37250277400016785, -1.7563471794128418, 0.9979022741317749, 1.0083775520324707, 1.1874009370803833, 0.3483320474624634, -1.2429407835006714, -0.5214757919311523, 1.0282204151153564, 0.910351574420929, 0.663429856300354, 2.332803249359131, 1.1575454473495483, 0.28762179613113403, 0.33196502923965454, -1.6719200611114502, -0.4797084331512451, 1.0305932760238647, -1.685714602470398, 0.3847081959247589, 1.0068269968032837, -0.2549194395542145, 0.6547319889068604, -0.9110058546066284, 1.8066890239715576, -0.32494980096817017, -2.22104549407959, 0.48271414637565613, -0.44328951835632324, 1.7001699209213257, 0.7447333335876465, -0.332958847284317, 1.6035118103027344, -0.31005388498306274, -1.2931705713272095, 1.1548786163330078, 1.435874581336975, -0.2515559494495392, 1.0147546529769897, 1.8394497632980347, -0.6663856506347656, -3.0723650455474854, 0.5602816343307495, -0.26233211159706116, 0.9346306920051575, 0.20490597188472748, -0.43416303396224976, -1.0380455255508423, -1.196144700050354, -0.5813020467758179, -0.9657883048057556, 0.7852985858917236, 0.6924593448638916, -0.6450380682945251, 1.2046416997909546, -0.37650641798973083, -0.5151007771492004, 0.8723293542861938, 0.2688109576702118, -0.5946903824806213, 0.8357033133506775, 1.6774719953536987, -1.326357364654541, -0.2770423889160156, -2.1274282932281494, -0.11611121892929077, 2.4799444675445557, 0.24039891362190247, -0.15748091042041779, 0.165469229221344, -1.0034867525100708, 0.1675804853439331, -0.37217676639556885, -0.7915287613868713, 0.8767434358596802, -0.23334427177906036, 0.24805645644664764, -0.6702320575714111, 0.1534573882818222, 0.5754225850105286, -0.0454661063849926, 1.5742123126983643, -0.03882389888167381, 0.3917081952095032, -0.10427918285131454, 0.9875443577766418, -0.5897349119186401, -1.0030064582824707, 0.05277993530035019, -0.34310024976730347, -0.6390401124954224, 1.59091055393219, -0.2796269953250885, 0.2706124484539032, -1.2496342658996582, -0.8579272031784058, -0.24972514808177948, 1.5186522006988525, -0.4051588773727417, 0.26406240463256836, -1.9822174310684204, -0.6285159587860107, -0.2439245730638504, 0.85382479429245, 0.5223366618156433, -1.627261757850647, 1.0006048679351807, 0.6793947815895081, 0.805046796798706, -1.1538655757904053, 1.712110161781311, -1.8747910261154175, -0.07595037668943405, -0.006566543597728014, 0.5384082794189453, 0.1852235347032547, -0.5931646823883057, 1.6209789514541626, -1.9649139642715454, -0.7306362986564636, -0.43680208921432495, 1.0307756662368774, 0.3297552466392517, -1.7320568561553955, 2.1949636936187744, -0.10724275559186935, 1.236384630203247, 2.0665059089660645, -0.3993922472000122, 0.34577473998069763, -1.5216619968414307, -0.7062160968780518, 1.5923476219177246, -0.2535894215106964, 0.9934040904045105, -0.0008085927693173289, 0.7341535687446594, 0.6852608323097229, 2.127528667449951], "dtype": "float32"}], "e": -32.6790771484375}, {"a": [{"__nd__": [0.6071195602416992, 0.19704949855804443, 0.6166778802871704, -0.5855146646499634, -1.044865608215332, -2.5348751544952393, 0.710246205329895, 0.278719425201416, -0.33464208245277405, 0.9471038579940796, 0.41188856959342957, -0.2888810634613037, 0.7585325837135315, 0.6210304498672485, 0.8051108717918396, 1.2000237703323364, 0.42180317640304565, 0.7601162195205688, -0.049792904406785965, -0.9464503526687622, -0.15892569720745087, 0.06633085012435913, -0.9664117693901062, -0.03508024290204048, 0.325582891702652, -0.8989149332046509, -0.0933072492480278, 0.2325356900691986, -0.03126698359847069, 1.2781516313552856, -1.0497612953186035, 2.1979217529296875, -0.45008355379104614, 1.7651268243789673, 0.14318619668483734, 1.9663640260696411, -0.028649412095546722, -0.25067397952079773, -1.2069636583328247, -0.553794264793396, 0.7471497058868408, 0.16947737336158752, 0.22760511934757233, -0.014973782002925873, 0.0035991580225527287, 0.7034088373184204, 0.15551742911338806, -0.7687461376190186, 1.1623964309692383, -0.019704090431332588, -0.09390893578529358, 0.21400228142738342, -0.49436432123184204, -0.58463454246521, -1.046531081199646, 0.9956299662590027, -1.471972942352295, 0.21576835215091705, -1.6600092649459839, -1.4144660234451294, 1.0479010343551636, -0.01572592556476593, 0.663497269153595, 0.05838211625814438, -0.08360663056373596, 0.8615694046020508, 0.30932775139808655, 1.1490862369537354, 0.8770921230316162, -0.9397209286689758, -1.2050116062164307, 1.4189544916152954, 1.1390275955200195, -0.6030405163764954, 1.0336945056915283, -1.2002370357513428, -1.475260615348816, 1.1361209154129028, -0.34511539340019226, -1.2604749202728271, -1.0738420486450195, 1.2106043100357056, 1.7551299333572388, -0.33138152956962585, 1.7775160074234009, -1.1008713245391846, -2.731455087661743, -0.25277456641197205, 1.268092155456543, 0.15125279128551483, -0.46961531043052673, 0.7825441360473633, -0.3075309097766876, 0.3024362623691559, -1.1625028848648071, 0.7993832230567932, -0.8806561827659607, -0.553534746170044, 1.8498036861419678, 0.07947053760290146, 1.740967869758606, -1.416321873664856, -0.06649430841207504, 0.39741191267967224, 0.9824857115745544, 1.3888260126113892, 0.6774786114692688, 1.1263507604599, 0.20430222153663635, 1.1158065795898438, 1.0379375219345093, 0.525823712348938, 1.1917580366134644, 0.15815246105194092, -0.08583665639162064, -0.7939035892486572, -0.40262293815612793, 0.5189236402511597, -1.7977867126464844, 0.5054013729095459, -0.41091471910476685, 0.9371485710144043, -0.8959159851074219, 0.09725210815668106, -1.158789038658142, -0.5634096264839172, 0.3579981327056885, 1.8153200149536133, 0.5786710977554321, 0.1989629864692688, -1.2299034595489502, 0.5508721470832825, -0.8805345296859741, 1.012731671333313, 0.756611168384552, 1.001692533493042, -2.4853720664978027, -1.2508468627929688, -0.8015244603157043, -0.2810680568218231, -0.2869461476802826, -0.21928240358829498, 0.5172350406646729, 0.9129448533058167, -0.4352657198905945, -1.136802315711975, -0.059969909489154816, 0.17721962928771973, 0.10279609262943268, -1.202246069908142, 0.6290618181228638, 0.38278746604919434, 0.42487290501594543, -1.0925333499908447, 0.8960049152374268, 0.05647232383489609, 0.4233238697052002, 0.38107243180274963, -1.0497469902038574, -0.1809336245059967, 0.7912378311157227, 0.12826238572597504, 0.36390647292137146, -1.0496203899383545, -1.4869182109832764, -0.18939697742462158, -2.620781183242798, 0.9017279744148254, -0.8926705121994019, 1.6804245710372925, -0.8652650713920593, -0.2745993733406067, -1.3971712589263916, -1.5619093179702759, 0.12766452133655548, 0.4528144598007202, -1.5506374835968018, 0.42257338762283325, -0.7330893874168396, -1.041651964187622, 0.0211202260106802, 0.9861093759536743, -0.43572139739990234, -0.7277911305427551, 0.06726851314306259, -0.8363290429115295, 0.4734715223312378, -0.7442473769187927, -0.2986718714237213, -2.1217007637023926, -0.7605786919593811, -1.83327054977417, 1.3384636640548706, -0.9186661839485168, -0.07668407261371613, -0.0032401930075138807, -0.48192450404167175, -0.5758500099182129, 0.6408179402351379, 1.0630570650100708, 0.7214077711105347, 0.15529491007328033, -0.6596687436103821, -0.06370434165000916, 0.39099669456481934, 1.6634012460708618, 0.26543569564819336, -0.8139528632164001, -0.8465073704719543, -0.29209667444229126, -0.23465390503406525, 1.45198655128479, 0.3173341155052185, 0.6899827718734741, -0.6467213034629822, -1.3964643478393555, 2.541614532470703, 2.204329252243042, 0.6929527521133423, 0.9360966682434082, -0.6193339824676514, 1.4611948728561401, 3.1569602489471436, -0.046724606305360794, 0.4394327700138092, 0.9037783741950989, -0.38554567098617554, -0.14864112436771393, 0.23492972552776337, -0.8072123527526855, 0.5848509073257446, -1.1828973293304443, -0.014107886701822281, -0.10950377583503723, 0.3572444021701813, -0.5893874764442444, -1.4766398668289185, -0.3163427412509918, -0.17100943624973297, 1.2770618200302124, -0.0783613994717598, 0.806633710861206, -0.03054894506931305, -0.007523767184466124, -0.3883322775363922, -1.6378031969070435, -1.5282025337219238, -2.0158679485321045, -0.31437447667121887, 1.542969822883606, -1.4327363967895508, 0.9232386946678162, 1.842758059501648, 0.054248906672000885, 0.7799904942512512, -0.2586688995361328, -0.7911655306816101, -0.812052845954895, -0.6432390213012695, 0.5790032148361206, 0.24836522340774536, -0.6889222860336304, 1.0887171030044556, 0.22866518795490265, -0.9650619626045227, 0.2979395389556885, 1.517079472541809, 1.4032775163650513, -0.9799947142601013, 0.5917204022407532, 0.15825265645980835, 1.3284801244735718, 2.08381724357605, -2.5924205780029297, 0.8778766989707947, -1.3032251596450806, 0.20635300874710083, 1.5169668197631836, 1.4434062242507935, -0.6283194422721863, 1.7331353425979614, -0.4208543002605438, 0.02488211914896965, -1.2514623403549194, -0.5397392511367798, -1.1718950271606445, 0.6901071667671204, -0.7205732464790344, 1.0165789127349854, -0.7178437113761902, 1.056443214416504, 0.7959837913513184, -0.09398514777421951, 0.6363402009010315, 1.4998165369033813, 1.6809550523757935, -0.24769021570682526, -0.036857396364212036, 1.1493393182754517, 2.991330862045288, 0.8192300200462341, -1.1057562828063965, -0.785120964050293, -0.2289498895406723, 0.1972513347864151, -1.394982933998108, -2.0438177585601807, 0.21095365285873413, -0.38066044449806213, 0.042292408645153046, -1.5674948692321777, -0.6355501413345337, 1.0317803621292114, -0.27360010147094727, -1.2071592807769775, 0.7275385856628418, -0.3809818625450134, 0.8983707427978516, 0.4204634130001068, -0.22708526253700256, -0.35783013701438904, 0.9888245463371277, -0.4603530168533325, 0.47519004344940186, 1.3033888339996338, -0.5825690031051636, -0.579861044883728, -0.04263896122574806, -1.440704107284546, 0.4115329384803772, -0.8974432349205017, -0.549907922744751, -1.4217718839645386, 1.182724952697754, 1.3955776691436768, 0.16229793429374695, 0.6352351903915405, 0.42804306745529175, 1.145529866218567, 2.1018269062042236, 2.518141269683838, 0.6457509398460388, 1.0516823530197144, -0.5964319705963135, 0.9128533601760864, 1.2807859182357788, -0.9854618906974792, -0.21814827620983124, -1.3106642961502075, -0.09909039735794067, -0.36060115694999695, -1.340904712677002, -1.7163009643554688, -0.289345920085907, -1.6274358034133911, 0.7655467987060547, -0.7104535102844238, -0.8558796048164368, 0.19449648261070251, 0.26431578397750854, -0.9830437898635864, 0.6997559070587158, 0.40430814027786255, 0.5141797661781311, 1.8866081237792969, -0.2655908465385437, 0.11848035454750061, -1.5296969413757324, 0.38844847679138184, -0.8960760235786438, -0.8394914865493774, -0.06129764765501022, 0.16891148686408997, -0.2208254188299179, -0.10184299945831299, -0.4335426688194275, -1.7646806240081787, 1.6410585641860962, -0.41656479239463806, 0.21518245339393616, -1.518311858177185, -0.7220835089683533, 0.3294828534126282, -0.3627592921257019, -0.9791452884674072, -0.348649263381958, 0.14451864361763, -0.5135028958320618, -1.2106003761291504, 1.1992164850234985, -0.08127933740615845, 1.9034699201583862, 0.3654417395591736, -1.5083569288253784, -1.0973623991012573, -0.9480374455451965, 0.531762957572937, -1.577677607536316, 1.4544732570648193, -0.5999159812927246, 0.5968150496482849, 0.9320157766342163, -0.9625105857849121, -0.6202647686004639, 0.2268364429473877, 1.1606686115264893, 0.8278002142906189, -0.22104768455028534, 0.9909376502037048, 0.042441919445991516, 0.675390899181366, -1.0047608613967896, 1.7045713663101196, -0.13729584217071533, -0.46637803316116333, -0.04071418195962906, 0.3641767203807831, 0.8129711151123047, -0.5388963222503662, -1.7812422513961792, 2.2038700580596924, -0.6260647177696228, 1.2154728174209595, -0.7066168189048767, -1.0844005346298218, 0.9658306241035461, -1.3548970222473145, -2.793846368789673, 0.9001177549362183, 0.6116235852241516, 1.0931438207626343, 0.32655155658721924, -0.8579747080802917, -0.9245076775550842, -0.07475101947784424, 0.23810555040836334, -0.3184776306152344, 0.3030039072036743, 1.190817952156067, 0.32604771852493286, 0.9999205470085144, -0.44505229592323303, -0.6467537879943848, -0.882649838924408, 0.43628764152526855, -0.6438306570053101, 0.33214935660362244, -1.018681287765503, 0.39740562438964844, 1.1961097717285156, 0.18787045776844025, -0.025065118446946144, -0.6964273452758789, -0.3783223628997803, -0.6475038528442383, 1.0070593357086182, 1.0706446170806885, 0.7784916758537292, 0.05910290777683258, 0.6984531283378601, 0.138973668217659, -1.638627290725708, -0.8672055602073669, -1.230565071105957, -0.07270534336566925, -0.62265944480896, 0.3524281680583954, -1.4018018245697021, 0.31492650508880615, 0.6477852463722229, -0.25701189041137695, -0.8938801288604736, -0.8307225704193115, 0.26295405626296997, 0.07570018619298935, -2.3295750617980957, 1.1287946701049805, 0.10809428989887238, 0.33628544211387634, -0.6782630085945129, 0.9517754316329956, 1.6609843969345093, -0.7861881256103516, 0.11753755807876587, -0.11799986660480499, -0.27504611015319824, -0.5685326457023621, 1.4488807916641235, -1.697214126586914, 0.2237318903207779, -0.12853479385375977, -1.4266220331192017, 0.23100215196609497, -1.3066563606262207, 0.8786899447441101, 0.8009322285652161, 0.7614970207214355, 1.594628095626831, -0.8584614992141724, -0.19146297872066498, 0.383685439825058, 1.9497700929641724, 0.6872007250785828, 0.3147844076156616, 0.625625729560852, 0.8096964955329895, -0.7571418881416321, 1.2104835510253906, 0.9958673715591431, -0.2605903148651123, -0.6903558373451233, -1.6632190942764282, 0.6805222630500793, -0.20774595439434052, -0.6520654559135437, -1.0645818710327148, -0.963960587978363, 0.7928403615951538, -1.3006091117858887, 0.8274747133255005, 0.2738214135169983, 0.34757864475250244, -0.09436173737049103, -1.2326785326004028, 2.5204334259033203, 1.1928331851959229, -0.050513915717601776, 0.19301746785640717, 0.1675616055727005, -2.5144500732421875, 0.686016321182251, -0.4324782192707062, -0.13663668930530548, -1.9381177425384521, -0.7909003496170044, -1.0606491565704346, 2.0109095573425293, -0.3756185472011566, -0.12135056406259537, 0.878609836101532, 0.0026454797480255365, 0.6174649596214294, 0.8369096517562866, 0.1075030267238617, -0.06544935703277588, -1.270142674446106, -0.7649624347686768, -1.7750725746154785, 1.2431244850158691, 0.18479549884796143, 1.4543286561965942, 0.23365287482738495, 1.1419123411178589, -1.2174657583236694, -1.5087292194366455, 0.11887046694755554, -5.185114059713669e-05, 0.8790128827095032, -1.185511589050293, -1.4235843420028687, -0.5446799993515015, 0.3503662943840027, -0.3136101961135864, -0.7123329639434814, -1.2050297260284424, -0.4371916949748993, 0.6640098690986633, -0.0556054450571537, -1.4373693466186523, -0.18768328428268433, 1.539602518081665, -0.016353467479348183, 1.4791100025177002, -0.6230537295341492, -0.23634634912014008, 2.2117929458618164, -0.8211514949798584, -0.18066242337226868, -0.08855587989091873, 0.6466615796089172, -0.03644924983382225, 1.9711763858795166, 1.374464511871338, 1.1286782026290894, 0.6431160569190979, 0.9756875038146973, -1.4563175439834595, 0.611524760723114, -1.0912368297576904, -0.09571636468172073, 1.590781569480896, 0.9681553840637207, -0.836461067199707, 0.4093855023384094, 0.2772321105003357, 1.0726336240768433, 0.11217337101697922, -0.9764983654022217, 0.9994815587997437, -1.7817898988723755, 0.18962635099887848, -0.38401392102241516, -0.29952701926231384, 1.6501442193984985, -0.799295961856842, -0.5253976583480835, 0.5971062183380127, -2.173722982406616, -1.2663629055023193, 0.9804845452308655, 0.018050845712423325, 0.015216651372611523, 1.8414579629898071, -1.5428544282913208, 0.17739985883235931, -0.2941805422306061, 0.2874787151813507, -0.9769291281700134, -1.9395439624786377, -0.09070462733507156, -1.4372590780258179, -0.5520985126495361, -1.4808437824249268, 0.5074270367622375, -0.44317999482154846, 1.0763323307037354, 1.413501262664795, -0.841251790523529, -1.0169140100479126, -0.073613740503788, 0.7060620188713074, 0.9884259700775146, -0.16711129248142242, 0.2851038873195648, 0.8659312129020691, 0.3336232602596283, -0.9870439171791077, 0.38867348432540894, 0.9210613965988159, -1.1419793367385864, 0.09208548814058304, 0.4139370024204254, -0.4662732779979706, -0.3019230365753174, 0.056280989199876785, 0.12180887907743454, 0.03983263298869133, -0.6509988307952881, -0.42103537917137146, 1.0204172134399414, -1.1962532997131348, -0.11442379653453827, -2.0386877059936523, 0.5467942357063293, -1.6231801509857178, 0.6182088255882263, 1.1446195840835571, -0.06710521876811981, -0.7210294604301453, 0.24002528190612793, 1.3207266330718994, 1.1834617853164673, -0.6902164220809937, 0.8278465867042542, -0.3418351709842682, 0.7007633447647095, 0.2642616033554077, 0.29252591729164124, 0.04307621344923973, -0.28548553586006165, 0.9930307865142822, -1.3137508630752563, -0.8899078965187073, -0.04787978529930115, -0.12161799520254135, 0.4674440920352936, 0.8749356269836426, -0.5315681099891663, -1.1440831422805786, 2.1431257724761963, -2.4316210746765137, 0.06472988426685333, 0.018962271511554718, -0.5870810151100159, 1.7372231483459473, 0.1360180675983429, -2.890993595123291, 0.3312104344367981, 0.6823692917823792, 1.6224583387374878, 0.20311753451824188, -1.2565118074417114, -0.5209063291549683, -0.6802037358283997, -1.5389381647109985, 0.2701238989830017, 0.7639499306678772, 0.9146004319190979, 0.9352496266365051, 0.44855600595474243, -0.5465964674949646, 2.0395004749298096, -0.03869931772351265, -1.2147321701049805, 0.16191929578781128, -1.1528953313827515, -0.08242221921682358, 0.6858585476875305, -0.725344717502594, -0.2846745252609253, 0.34587475657463074, -0.16374069452285767, -0.14573174715042114, -1.0327811241149902, -0.8962883949279785, 0.20766012370586395, -0.2746417224407196, 0.07166258990764618, -0.3295877277851105, -1.7814419269561768, 0.09281019866466522, -1.4854280948638916, 0.3934497833251953, 0.2605328857898712, -0.009649325162172318, -0.12648431956768036, 0.9685455560684204, 0.6352208852767944, -1.9956154823303223, -0.5189077258110046, 0.9298479557037354, -1.0570509433746338, -1.3656400442123413, 0.7078751921653748, -1.0030511617660522, 0.8977535367012024, 0.8334801197052002, -0.7069929838180542, -1.0790979862213135, 1.97157883644104, 0.43803641200065613, 0.023872142657637596, 0.17437143623828888, 1.5023937225341797, 0.4321359395980835, -0.46188223361968994, -0.2797142565250397, 0.9337159991264343, 0.4615671634674072, 0.0987139567732811, 0.4089140295982361, -1.8592449426651, 1.6577532291412354, -0.10653457790613174, -0.5807692408561707, -2.449101209640503, 0.32922279834747314, -2.2369749546051025, 1.8550575971603394, -2.7439534664154053, 0.3308888077735901, -1.0055773258209229, 0.9195401668548584, -2.987014055252075, -0.919691801071167, -2.008345603942871, -0.5516297817230225, -0.30103930830955505, -0.12546542286872864, 0.5845942497253418, -0.4049665331840515, -0.9534085988998413, 0.028855644166469574, -0.2825510799884796, -0.3267919719219208, 0.16367828845977783, 1.5063799619674683, 0.29122796654701233, 1.0353776216506958, 0.8097018599510193, -0.904245138168335, 1.4952051639556885, 0.5118302702903748, -0.4693796634674072, 0.4792896807193756, 1.1738414764404297, 0.5301425457000732, 1.1446832418441772, -0.579675555229187, -1.8673571348190308, -0.7441267967224121, -0.06780297309160233, -0.37130996584892273, 0.7163571119308472, -1.8334704637527466, 0.23833224177360535, 1.3120771646499634, 0.33921635150909424, -0.20173899829387665, 1.1073564291000366, -1.6902867555618286, 0.47081616520881653, 1.769302487373352, -0.4627348780632019, 0.3755268454551697, -0.04568837210536003, -0.0930081382393837, 0.6345847249031067, 0.9860129356384277, -1.7275447845458984, -2.675380229949951, 0.31453821063041687, -0.8300767540931702, -0.7624902129173279, -0.26997461915016174, -0.10714008659124374, -0.19947892427444458, 0.7992079257965088, -0.5398789644241333, 0.5727194547653198, -0.6731700897216797, -0.37595686316490173, -0.47384023666381836, 1.3977967500686646, 0.8391832113265991, 0.4865656793117523, -0.46070846915245056, -0.5857160687446594, 0.5170252323150635, 0.8881784081459045, 0.8077441453933716, -0.36793753504753113, -0.1133580282330513, -0.8090657591819763, 0.6700394749641418, 0.5506296753883362, 1.1568583250045776, 0.6543858647346497, 0.7235076427459717, -0.5930576920509338, -1.2514821290969849, 1.048543930053711, -0.01446268055588007, 0.8962880373001099, -1.8454594612121582, 0.13765393197536469, -0.5534802079200745, 0.5170350670814514, -2.7259395122528076, -0.20582576096057892, -0.5272553563117981, -0.8814441561698914, -1.2706488370895386, 1.4174726009368896, 0.12464523315429688, -0.1590413600206375, -0.11467268317937851, 0.5544068217277527, 0.5312567353248596, 1.164731502532959, -2.15470814704895, -2.415954351425171, -0.46808263659477234, -1.629983901977539, 0.6996193528175354, -0.5405165553092957, 2.2758429050445557, -0.13172109425067902, -1.210654616355896, -0.417194664478302, 1.1227025985717773, 0.48169681429862976, -0.027208596467971802, -0.7624778151512146, -1.464660406112671, -0.6078179478645325, 1.9526835680007935, 0.012278352864086628, -1.276833415031433, -1.6417865753173828, 0.20788206160068512, 1.2260743379592896, 0.8378313779830933, -0.7791594862937927, -1.589381217956543, 0.7109445333480835, -0.8474316000938416, 1.0213688611984253, -1.6406562328338623, -0.14287251234054565, -0.9879817366600037, -0.7219105958938599, 1.3995864391326904, -1.6085593700408936, -1.7460987567901611, 0.197146937251091, 0.6493005156517029, -0.3228251039981842, 1.2108285427093506, -0.05891799181699753, -2.2745115756988525, -0.30825456976890564, 1.0420231819152832, -0.5766682624816895, -0.6478269100189209, 1.077425241470337, 2.355875015258789, 0.36203527450561523, -0.24738256633281708, 1.4596036672592163, 0.5086103081703186, 0.7660743594169617, -0.1436673402786255, 0.8519015908241272, -0.14331284165382385, 1.776593565940857, 0.4772135615348816, -0.20276397466659546, 1.5079920291900635, -0.43151241540908813, -2.768070697784424, -1.3197237253189087, -0.702240526676178, 0.5381743907928467, 1.1204025745391846, -0.684666097164154, 0.8276179432868958, 0.17270101606845856, -0.5507444143295288, 0.23397091031074524, -1.8819252252578735, -0.7419625520706177, 1.1991204023361206, -0.9906664490699768, 0.7095429301261902, -0.44884318113327026, 0.11753516644239426, -1.6998473405838013, 0.7119666337966919, -0.39297354221343994, -0.9913907051086426, 3.6853199005126953, -2.2548716068267822, -0.013633404858410358, 0.2134999930858612, 1.447960615158081, -1.3433977365493774, 0.5470371246337891, 0.49682238698005676, 0.04145386815071106, 0.4491833448410034, -0.3138759732246399, -0.5920997858047485, 1.0249125957489014, -1.0933696031570435, 0.5383039116859436, -0.8160018920898438, -0.854456901550293, 1.4109890460968018, -0.21974694728851318, -1.252832293510437, -0.14944300055503845, 0.44156700372695923, -0.8985969424247742, 0.05960714817047119, -1.12476646900177, -1.8292768001556396, 0.9858478307723999, 0.556480884552002, 1.280951976776123, 2.2497169971466064, 1.869006872177124, 0.3913613557815552, -0.003649953519925475, -0.5528044700622559, 1.485594391822815, 0.3691784143447876, -0.14059391617774963, -1.032682180404663, -0.5538682341575623, -0.20514529943466187, -0.29678240418434143, -0.15235920250415802, 0.3981727957725525, 1.713661551475525, 0.7700684666633606, 0.3020694851875305, 0.4924173057079315, 1.063706874847412, 1.6445375680923462, -1.1987171173095703, -0.7652925848960876, 0.5322393774986267, 1.9405796527862549, 0.7440841197967529, 0.9316606521606445, -0.5687605738639832, 0.7279391884803772, 1.2262156009674072, -0.7418234944343567, 0.1019669845700264, 1.5914884805679321, 0.4174538850784302, -0.48814862966537476, 0.5442418456077576, -0.15367937088012695, 0.4179132878780365, 1.2324501276016235, -0.2627791464328766, 0.6068674921989441, -0.13958334922790527, -1.922338843345642, -1.480961799621582, 0.032851092517375946, 0.8956441879272461, 1.2792471647262573, -2.0438342094421387, 0.3844131529331207, -1.308240294456482, -2.6897547245025635, -0.4343372881412506, -0.9807626008987427, 0.8042153120040894, 0.22501695156097412, -1.2557278871536255, -0.7151502966880798, 1.1009529829025269, 0.43606218695640564, 0.7123855352401733, -0.8356465697288513, -1.0360655784606934, 1.2174590826034546, 1.1986020803451538, -0.39549028873443604, -0.9396800398826599, 0.6452674865722656, 0.5020885467529297, -1.917230486869812, 0.553814709186554, 0.7967376708984375, -0.706569492816925, 1.4824663400650024, 0.22280222177505493, 0.35062190890312195, -1.2635411024093628, -0.7993014454841614, -0.1479734182357788, 0.40178102254867554, 0.38381582498550415, -0.3743140995502472, 0.5317651033401489, -1.4187099933624268, -0.17311140894889832, -0.7166181206703186, -0.6713342666625977, 0.19521775841712952, 0.0400259904563427, 0.5738954544067383, 1.0375574827194214, -1.8177865743637085, 0.8555241227149963, -0.5161269903182983, 0.0873919352889061, -0.09010137617588043, 0.45392292737960815, -1.000287652015686, -0.988375186920166, -0.8556308746337891, -0.5066618323326111, 0.16166922450065613, 0.13954271376132965, -0.43425890803337097, 0.8544070720672607, -0.3041480779647827, -0.8834580779075623, 0.5243114233016968, -1.513137698173523, 0.46411868929862976, 0.8396427631378174, 0.05512627586722374, 1.8444722890853882, -0.5691289305686951, -1.7867801189422607, 1.4121299982070923, -0.012285196222364902, 0.5979064106941223, 0.12232238799333572, -0.6519571542739868, 0.635482132434845, -1.3919587135314941, 0.9053184986114502, -0.6093654036521912, 0.47569039463996887, 0.7363051176071167, 0.5057922601699829, 0.20600926876068115, 1.3521721363067627, 0.11533975601196289, -0.27920717000961304, 0.5232199430465698, -0.7836459279060364, -1.0839186906814575, -1.819690465927124, 0.444979190826416, -0.43690094351768494, 1.3951236009597778, -0.3671097159385681, 0.8415015339851379, -0.09541772305965424, -0.2958880662918091, 0.40383458137512207, -0.315032035112381, 0.26107504963874817, -0.8152453303337097, -1.2163418531417847, 1.945329189300537, 0.2153579443693161, -0.8330994844436646, 0.026545384898781776, -0.33403337001800537, 0.1728684902191162, -1.0172312259674072, 0.1949872225522995, 0.15330937504768372, 0.5044231414794922, -0.14686451852321625, -0.14129503071308136, 0.893325924873352, 1.866807460784912, 0.3223104476928711, 0.793786883354187, 0.6534175872802734, -0.3994144797325134, 2.280430316925049, -1.1220918893814087, -2.2211594581604004, 0.10486529022455215, -0.5453241467475891, -0.6952413320541382, -1.3816529512405396, 0.1917533129453659, -0.7267739176750183, -0.5240300297737122, -1.252579689025879, 1.1459875106811523, -0.8184098601341248, -0.3017547130584717, -0.412869930267334, -0.5278530120849609, -0.056067902594804764, -0.9033010601997375, 1.3398771286010742, -0.04786703735589981, -1.205384612083435, 0.5544936656951904, 1.1077754497528076, -1.1651358604431152, -1.600757122039795, -0.4100198745727539, -0.3303373157978058, 0.23936161398887634, -1.2000772953033447, 0.9578765630722046, 0.40108901262283325, -1.7437890768051147, 0.07993605732917786, -0.7821558117866516, -0.667003870010376, -1.0435227155685425, 1.5891886949539185, 0.5502156019210815, -0.5531061887741089, -0.6890661120414734, -1.9217793941497803, 2.7412831783294678, -1.1137052774429321, 1.33485746383667, 0.16329053044319153, -0.7223320603370667, 0.5968283414840698, -0.5297213196754456, -0.2651795446872711, 0.8817091584205627, -1.1006172895431519, 1.5501718521118164, -2.141698122024536, 1.0087001323699951, -0.3018442392349243, -0.90823894739151, -1.6767961978912354, -0.8715124726295471, 0.12021441012620926, -1.0805189609527588, -0.03402429446578026, 0.9261064529418945, -0.7288756966590881, 0.09265382587909698, -1.1472594738006592, -0.16829805076122284, 0.8453871607780457, -1.7999387979507446, -1.1345889568328857, 0.5921046733856201, 1.3125379085540771, 0.527726948261261, 0.70735102891922, -0.37011846899986267, -0.8266396522521973, 0.5833847522735596, -1.0791922807693481, 0.17450687289237976, -0.8545259237289429, 0.42681947350502014, -0.3375859260559082, -1.9102520942687988, 0.2897191047668457, 0.04340806230902672, -0.6264861226081848, -0.605582594871521, -0.6587537527084351, -2.160846710205078, 0.49202054738998413, 0.11773443967103958, 1.7644621133804321, 1.702943205833435, 0.297986775636673, -2.527507781982422, -0.8546380400657654, 0.44716715812683105, -1.0807541608810425, 0.9827443957328796, 0.40237733721733093, -0.08920023590326309, -0.9188770055770874, 0.6519980430603027, 0.2970599830150604, -1.6562854051589966, 0.5989683270454407, -1.4825884103775024, -0.1189294084906578, 2.1057229042053223, -1.164811372756958, 1.1842389106750488, -2.0426502227783203, -1.7007910013198853, -1.586093783378601, -1.0283998250961304, -1.4157671928405762, 0.3895587623119354, 1.1136854887008667, -0.6645370125770569, -1.3160799741744995, -0.47760409116744995, -0.5013994574546814, -1.6537331342697144, -0.529061496257782, -0.5556430220603943, 0.14431732892990112, 1.5189523696899414, -0.2583164572715759, -0.0012115424033254385, 0.7665322422981262, 1.9138622283935547, -0.19776101410388947, -0.853452205657959, 0.1264382153749466, 0.8438195586204529, 0.23990963399410248, 0.9555797576904297, -0.4420672655105591, -0.2928389310836792, -0.8480648398399353, -0.032104477286338806, -0.5079609751701355, -0.3768060505390167, -1.538552165031433, -0.23916223645210266, -0.41948142647743225, -0.5818213224411011, -0.3432796895503998, -0.6642484664916992, 0.20761097967624664, 0.15602581202983856, -1.0521645545959473, 0.4540446400642395, -0.569907546043396, 0.026387814432382584, 0.5437865257263184, -1.3049861192703247, -1.7286311388015747, 0.5388877987861633, -0.003930581733584404, 1.911411166191101, 0.2719371020793915, -0.2736084461212158, 0.5751686692237854, 0.20190003514289856, -0.1503310203552246, 1.8987631797790527, 0.5435423254966736, -0.3369433283805847, -0.7870357036590576, 0.5166645646095276, 0.34853100776672363, 0.03530103713274002, 0.32374557852745056, 0.6337767839431763, 1.3374955654144287, 2.234407663345337, -1.5506891012191772, -0.8229464292526245, -0.5896424651145935, 0.4950243830680847, 0.8273321390151978, 2.0619609355926514, -0.3945068120956421, -0.20524975657463074, 0.10099924355745316, 0.12628549337387085, -1.0279918909072876, 0.3666696846485138, 0.4871107339859009, -0.6277923583984375, -0.435286283493042, 1.6522964239120483, -1.7328509092330933, 0.3408361077308655, 1.0534090995788574, 0.7587377429008484, 0.6460875868797302, 0.5694574117660522, -0.22434678673744202, 0.41470834612846375, 0.749870777130127, 1.3741953372955322, -0.9874124526977539, -2.333310604095459, -0.6504513025283813, -0.9362586736679077, 0.10787416994571686, -0.2612449526786804, -0.1023799329996109, 1.2873479127883911, -0.9872163534164429, 0.20117244124412537, -0.06583000719547272, -0.6306220293045044, -0.7523186206817627, -0.7903218865394592, 0.4075376093387604, 0.011683971621096134, 0.7609084248542786, 2.212221622467041, 0.6277311444282532, -0.22276028990745544, 0.265487402677536, -0.7330288290977478, -0.9647889733314514, -0.3678838908672333, 0.3937736451625824, -1.015801191329956, -0.23336726427078247, -2.182546615600586, -1.1770825386047363, 1.3724546432495117, 0.6148836612701416, 2.3493762016296387, 0.05677812919020653, 0.6667866110801697, 0.8042877316474915, -0.8398296236991882, -0.11372868716716766, 0.9525001049041748, -0.8432363271713257, 0.24052998423576355, 2.1081604957580566, -0.24943402409553528, -0.5261008739471436, 0.25133630633354187, -0.06857164204120636, 2.1694514751434326, -1.38306725025177, -0.8558155298233032, 1.2185750007629395, -0.2896695137023926, -0.10112209618091583, 2.0687179565429688, -1.3309720754623413, 0.43999823927879333, 0.39060911536216736, -0.6542622447013855, -0.6814551949501038, -0.1198984831571579, 0.8924473524093628, -0.6184234023094177, -1.708210825920105, -0.2671971023082733, -0.8237184882164001, 1.1770092248916626, 0.8355988264083862, 1.194920539855957, -0.8174024820327759, -1.0087337493896484, 1.336510181427002, 0.017175689339637756, 0.20724578201770782, 0.8155675530433655, -0.49701741337776184, 1.051774501800537, -1.2432156801223755, -2.1242153644561768, -1.1826895475387573, 0.8343778252601624, 0.3771173059940338, -2.8433003425598145, 0.8757386803627014, -0.9404319524765015, -1.060814380645752, 1.174782156944275, -0.01055082492530346, 0.9479125142097473, -2.5089221000671387, -0.21988247334957123, -1.5804997682571411, 0.48102450370788574, -0.18402419984340668, 0.4412972927093506, 1.9821926355361938, 0.6744983196258545, 0.18815398216247559, 1.0619627237319946, 0.6048644781112671, 0.7579985857009888, 1.1272096633911133, 1.7968991994857788, -0.8299330472946167, 0.7830402851104736, 0.027500005438923836, -0.2129366546869278, -0.7018926739692688, -0.5975978970527649, -1.030315637588501, 0.8712288737297058, -0.44978687167167664, 0.05483505129814148, 0.007496881298720837, -0.03132955729961395, 0.418530136346817, -0.2935088574886322, 0.888157844543457, -1.5560665130615234, -0.2663152813911438, -0.3275589644908905, -1.7336030006408691, 0.24083039164543152, -1.586462140083313, -1.496389389038086, 0.16863898932933807, -0.6358607411384583, 0.033074263483285904, -1.230404257774353, -0.9935377240180969, 1.615552306175232, -0.18083639442920685, -1.1316176652908325, -0.10802628844976425, 1.5581083297729492, 0.19396963715553284, -0.0854860246181488, 0.2621006667613983, 1.2076812982559204, 1.2030137777328491, -1.0601820945739746, 1.8035295009613037, 1.7622730731964111, -0.4822918772697449, 0.8547775149345398, -1.5555176734924316, 0.9682261347770691, -1.6411205530166626, 0.4521085023880005, -0.8023101687431335, 0.002138411859050393, 0.622081995010376, 0.6482009291648865, -0.4096372723579407, 0.2310863435268402, -0.25641608238220215, -1.5659936666488647, -0.4170857071876526, 0.5256087779998779, -0.8178834915161133, 0.33131465315818787, 1.0478461980819702, -1.1448417901992798, 0.471747487783432, 3.510082960128784, -0.3895098567008972, 1.059821367263794, -0.7715762853622437, 0.08847645670175552, -1.2404149770736694, 1.3739209175109863, -1.4423717260360718, -0.6597438454627991, -0.0023736038710922003, 0.0700654685497284, -1.7918665409088135, -0.1896667331457138, -0.3360452950000763, 0.4278910160064697, 0.22223927080631256, -1.6794633865356445, -1.1728456020355225, 0.7384211421012878, 1.2733750343322754, 1.1314486265182495, -1.9075241088867188, -0.6119580268859863, -1.2318229675292969, -1.3318231105804443, 0.7029541730880737, -1.5623936653137207, -0.5043538808822632, -1.3706620931625366, 1.0008374452590942, -0.6312894821166992, 0.17788156867027283, 0.6922481060028076, 0.06421640515327454, -1.462509036064148, 0.023854313418269157, -0.7950726747512817, -0.47473761439323425, 0.9790811538696289, 0.11969389021396637, -1.6331703662872314, 0.1034255400300026, 1.8639267683029175, 1.2520434856414795, 0.5093927979469299, 0.7555137276649475, 0.3926474153995514, -0.9713026285171509, -1.3112342357635498, 0.9599415063858032, -0.06118455529212952, 0.40602415800094604, 0.2617086172103882, 1.5109987258911133, 1.296734094619751, 0.7656170129776001, 0.7212950587272644, 0.7563468217849731, -0.34475550055503845, -1.476562261581421, 0.1935223937034607, 0.05472852289676666, 0.24337422847747803, -0.023051928728818893, 0.6796937584877014, 0.6221762299537659, -1.4224517345428467, -0.6307809948921204, 0.5349427461624146, 1.1844826936721802, 1.9890044927597046, -0.42469489574432373, 0.04868398234248161, -0.8725873827934265, 0.6630917191505432, 0.08140147477388382, -1.0007309913635254, 0.8892508745193481, 2.0019054412841797, 1.3645615577697754, -0.13355018198490143, 0.6909216642379761, -0.401017963886261, 2.41121244430542, -0.013806775212287903, -0.0990886315703392, -1.0336650609970093, 0.9707028269767761, -0.3947544991970062, -0.17522181570529938, -1.7452572584152222, -0.46918073296546936, 0.26241257786750793, 0.5903447866439819, 0.7395341992378235, -0.4640769362449646, 0.5989963412284851, -0.7424906492233276, -0.876486599445343, -0.7071289420127869, 0.4372088313102722, -0.19258536398410797, 0.7637020945549011, 1.8314523696899414, -0.03353988751769066, 0.32447606325149536, -0.40924713015556335, -1.031335711479187, 0.009081229567527771, -0.0015674785245209932, -0.8935661315917969, -0.7159764170646667, 1.1612887382507324, -0.28397271037101746, 0.0731288269162178, 0.9413196444511414, 0.34655797481536865, -0.7843279242515564, -1.2405263185501099, 0.44272103905677795, -0.06450540572404861, 0.028245076537132263, -0.17019255459308624, 1.1110948324203491, 0.17450052499771118, 1.914508581161499, -0.08198834210634232, 1.3233126401901245, 0.9734650254249573, -0.18197982013225555, 0.9291749000549316, -0.9766520261764526, -1.6458684206008911, 1.0037938356399536, -0.34964582324028015, 1.9441837072372437, -0.808899998664856, -1.6841139793395996, 1.575365662574768, -0.5840596556663513, 0.6880357265472412, -0.2880604565143585, 1.627825379371643, -0.9151867628097534, 0.037982501089572906, -0.9825182557106018, 0.9679804444313049, 0.5626018643379211, -0.2382282018661499, 0.43229639530181885, -0.23058411478996277, -0.7622599005699158, 1.3279273509979248, 0.4687272310256958, 1.9339739084243774, 0.20999138057231903, -0.38184884190559387, 0.8699910640716553, 0.45239052176475525, 1.3522130250930786, 0.2946864068508148, -0.33883681893348694, 0.798978865146637, 0.34859001636505127, 0.6203727722167969, -0.34138748049736023, 0.041043471544981, 0.2383618950843811, 0.4077887237071991, 1.0590488910675049, -0.08088107407093048, 0.7733086347579956, -0.3556542694568634, -0.06795083731412888, 0.5507110357284546, 0.660964846611023, 0.4419025182723999, 0.05724592134356499, -0.12750786542892456, -1.150476336479187, -1.1251989603042603, 0.6214679479598999, 0.027146030217409134, 1.695289134979248, 1.4103481769561768, -0.7862759232521057, 0.8375483155250549, 0.6522695422172546, 0.8079839944839478, 0.5684571862220764, -0.9560689926147461, 2.468705415725708, 0.2922489643096924, 1.0697249174118042, -0.9979224801063538, -0.5150633454322815, 0.6243307590484619, 0.9099122285842896, 0.7100536227226257, 0.116682268679142, 1.2635523080825806, -0.16671296954154968, -0.6486054062843323, 0.6958195567131042, -0.9175712466239929, -0.33649301528930664, -0.46257567405700684, -0.5684583783149719, -0.5152717232704163, -0.32699692249298096, -1.072037696838379, 0.8291050791740417, -0.13653267920017242, -0.2045975774526596, 0.9069164395332336, -0.7878402471542358, -1.0394450426101685, -0.16516387462615967, 0.7436912655830383, -0.6681089997291565, -0.5586323738098145, 0.8801447749137878, -1.518833875656128, 1.2873551845550537, -0.8382090330123901, -1.2876948118209839, -0.38852041959762573, -0.46745896339416504, -0.8983098864555359, 0.4126022458076477, -2.104694128036499, -0.36808785796165466, -0.4778728783130646, -1.818631887435913, -0.8073084354400635, 1.1567208766937256, -0.4550360143184662, -0.1174481138586998, -0.6271223425865173, -0.2561537027359009, -1.6598446369171143, 1.0686688423156738, -1.354898452758789, -0.8402557373046875, 1.2898011207580566, 0.04451246187090874, 1.8787060976028442, 0.8975175619125366, -0.34970465302467346, -0.7735461592674255, -0.18892568349838257, -0.3463789224624634, 0.0956801250576973, 0.9466780424118042, 0.5542072057723999, -0.9759456515312195, 1.1682994365692139, 0.586832582950592, -0.061656855046749115, 0.7496895790100098, 0.5708644986152649, 0.8107461929321289, -0.7843319177627563, 0.024922406300902367, 1.921331763267517, -1.1257529258728027, -0.8023388385772705, 1.8730164766311646, -0.12181730568408966, 1.1381711959838867, -0.6079328656196594, -0.1386498361825943, 1.6733063459396362, 1.9315125942230225, 0.5472313761711121, 0.4217137098312378, 1.4776374101638794, 0.6955711841583252, 0.38445577025413513, -1.4469807147979736, -0.6852988004684448, -0.9291775226593018, 1.1713371276855469, -0.05734605714678764, 0.22305132448673248, 0.47654077410697937, -0.024760546162724495, -0.030070841312408447, -0.5822666883468628, 0.06934492290019989, -0.23547975718975067, -0.2619716227054596, 2.8429641723632812, 0.6974210739135742, -1.661973476409912, -0.7299947738647461, 0.5624651312828064, -0.7983092665672302, -0.23005186021327972, 0.4236089587211609, -0.8565142154693604, -0.9875094890594482, -0.40044301748275757, 0.3860464096069336, -1.3588308095932007, -0.8886849284172058, 1.3466311693191528, 2.8065574169158936, 1.3591445684432983, -0.7776795625686646, -0.10101401805877686, -1.7897027730941772, 1.7857283353805542, 0.7398452758789062, -0.09439435601234436, -0.7114977836608887, -0.21738310158252716, -1.878353476524353, -0.6926836967468262, -2.105560541152954, -2.775569438934326, 1.7315553426742554, -0.2846912741661072, 0.44474852085113525, 0.33301714062690735, -0.2528652250766754, 0.8900429010391235, -1.0016597509384155, -0.41342368721961975, -0.9617526531219482, -1.0911701917648315, -1.5831471681594849, 0.8951738476753235, -0.9421324729919434, 0.6183803081512451, -1.552412509918213, -0.6283299922943115, 0.6696581840515137, -0.8839423060417175, 1.521078109741211, -1.5276859998703003, 0.4055691063404083, -0.3053164482116699, -1.317354440689087, 0.16636809706687927, -0.3571273684501648, 1.2332844734191895, 0.9470223188400269, 0.30140942335128784, 1.0055099725723267, 1.0141273736953735, -1.8645890951156616, -0.8347505331039429, 0.19450731575489044, -1.2211652994155884, -0.990760326385498, -1.7439308166503906, 0.8853961825370789, 0.9148736000061035, 0.18326948583126068, 0.9994587302207947, -0.19991418719291687, -0.32776227593421936, 0.35933002829551697, 0.8887909054756165, 0.8039705753326416, 0.8958138227462769, 0.8203479051589966, -0.7190584540367126, 0.0127862012013793, -0.2172163724899292, 0.16386279463768005, 1.6471927165985107, -0.9494391083717346, -0.5394582152366638, 1.4214760065078735, 0.46569791436195374, 0.3209002614021301, -0.13131634891033173, -2.0785024166107178, 0.07527679949998856, -0.056744109839200974, 0.8258835077285767, -1.4734578132629395, 2.114589214324951, -0.5789361596107483, 0.47154125571250916, 0.22700689733028412, 0.20087088644504547, -0.12759819626808167, -0.07730231434106827, 1.713394284248352, -0.6412891745567322, 1.4717155694961548, -0.05447562411427498, -0.9553758502006531, -1.8152700662612915, -0.6659111380577087, -0.16213130950927734, -0.05633225291967392, 1.2829509973526, 0.06877020001411438, -1.0843160152435303, 0.4736628234386444, 0.4873802959918976, -0.7343127727508545, -1.3792802095413208, 1.337969183921814, 1.26486074924469, -0.3193679451942444, -1.1113147735595703, -1.908980131149292, 1.3552680015563965, -0.7086384296417236, 0.24809043109416962, -1.3791980743408203, 0.9274131059646606, -1.9908443689346313, -1.2183037996292114, 0.06465405970811844, -0.08146105706691742, 1.521880030632019, 1.3152917623519897, 0.9056384563446045, 1.7210688591003418, 2.574521541595459, -0.8889387845993042, 2.4402904510498047, 0.7417458295822144, 0.8991692662239075, 0.7230777144432068, -0.3326978087425232, 0.693537712097168, 1.4851679801940918, -0.7619814276695251, -0.575129508972168, 0.31411266326904297, -1.0313085317611694, 1.5112704038619995, 0.15584927797317505, -0.20847240090370178, -0.95191490650177, 0.662081778049469, 0.32471656799316406, 0.44184160232543945, 0.43683183193206787, -0.7481099367141724, -1.1617449522018433, 0.1218770295381546, 0.8089566230773926, 0.7664545178413391, 0.9429711699485779, 0.36873310804367065, 0.23902486264705658, 1.2283437252044678, -0.23314833641052246, -0.6567867994308472, -0.8908351063728333, -1.809692621231079, -1.756422996520996, 0.9583587050437927, -0.7947681546211243, -0.8095625638961792, 1.6809900999069214, -0.9175094366073608, -0.12303934246301651, 0.251932293176651, -0.3215644955635071, 1.1180163621902466, 1.0075476169586182, 0.4062248468399048, -0.39396145939826965, 0.7120704054832458, 0.40739205479621887, 0.7617993950843811, -0.2592370808124542, -1.221488356590271, -1.7587509155273438, -0.7106030583381653, -2.1878292560577393, 1.3333922624588013, -0.8352335095405579, -0.7383095622062683, -0.08786427229642868, -0.6057317852973938, -0.5200617909431458, -2.5897233486175537, 1.174796462059021, 0.16471783816814423, 1.158782720565796, 0.4858153462409973, 0.9100660085678101, -1.2241542339324951, -0.7017356753349304, -0.594049334526062, -0.731182873249054, -0.7641835808753967, -0.08409267663955688, -1.2352972030639648, 0.23573856055736542, -0.14205355942249298, -2.0753936767578125, -0.7595346570014954, 0.7024144530296326, -0.46796196699142456, -0.6702159643173218, -0.4873356819152832, 0.7625409960746765, -0.6967312097549438, 1.1511465311050415, -1.3957710266113281, -0.58983314037323, -0.9606017470359802, 0.8778109550476074, 0.7535267472267151, 0.25714507699012756, 0.2641151249408722, -1.5135973691940308, -0.9674607515335083, 0.34625303745269775, -0.1291104406118393, 1.0673165321350098, -0.572548508644104, 0.4544501006603241, 0.6166338920593262, 1.571969985961914, 1.9246325492858887, -0.3908078372478485, -1.1210107803344727, 0.5924087762832642, -1.0943946838378906, 0.18771708011627197, -0.37267234921455383, 1.2833809852600098, -2.5758655071258545, -0.8786184191703796, -0.08562372624874115, 0.2772049605846405, 0.6374385356903076, -0.8087888360023499, -0.19769912958145142, 0.7275460958480835, -0.227711021900177, -1.568715214729309, 1.6407300233840942, 0.23076777160167694, 0.9012777805328369, -0.0566725917160511, 0.2640454173088074, -1.6472628116607666, -0.7056837677955627, -2.272455930709839, -0.5075734853744507, 0.8474271893501282, -0.9503804445266724, -1.1110748052597046, 2.068535804748535, -1.1095353364944458, 0.25612199306488037, 0.7698807120323181, 0.22788836061954498, 1.2339648008346558, 0.37218233942985535, -1.5964977741241455, 0.30736789107322693, -1.3739508390426636, 0.1548643857240677, -0.01747165434062481, 0.5974101424217224, -1.1479427814483643, 0.44817760586738586, 0.8590940237045288, 1.2286163568496704, -2.5162441730499268, -0.14738285541534424, 0.89799964427948, 1.316522240638733, 1.0036362409591675, -0.2973581850528717, -2.185288667678833, -0.5136387944221497, 1.2996630668640137, -1.8779699802398682, -0.19140227138996124, -1.3562065362930298, -0.5259587168693542, -1.1159495115280151, 0.4557838439941406, 0.46543797850608826, -0.1418692171573639, 0.9215940833091736, 0.5058717727661133, -0.1847008317708969, 0.010737013071775436, -0.6202735304832458, 1.7176916599273682, 0.27885791659355164, -0.6526902914047241, -1.6098558902740479, -1.6150929927825928, -0.3535007834434509, -0.7893902659416199, -1.1435723304748535, 0.6251697540283203, -0.674072802066803, 1.0865426063537598, 1.1771388053894043, -0.8800012469291687, -1.614526629447937, -0.32976245880126953, -0.2485484480857849, -0.6907888650894165, -0.3099561333656311, -1.2403984069824219, 0.7300627827644348, 2.074772596359253, -0.5394901037216187, 0.022129425778985023, -0.31688833236694336, -0.9896240234375, 1.7132577896118164, -0.8296521902084351, 0.023023050278425217, 2.5616443157196045, 0.37346646189689636, 0.08282676339149475, 1.3543460369110107, 0.47910723090171814, -1.3930315971374512, -0.8788658380508423, 0.14355014264583588, 1.0863488912582397, 0.3171640932559967, 0.9866504073143005, 0.27040377259254456, 0.8092713952064514, -0.32653549313545227, -0.8007619380950928, 0.39080411195755005, 0.38100844621658325, 0.36892491579055786, 1.2318029403686523, -0.24877645075321198, 2.0554563999176025, 1.1621642112731934, 0.2883239984512329, 0.4499598741531372, -0.14556850492954254, 0.5313855409622192, -0.791626513004303, 0.7569838762283325, -1.4167931079864502, -0.5606974363327026, 1.7018446922302246, 1.5120997428894043, -1.4982751607894897, 0.29476404190063477, 1.224263072013855, 1.2544708251953125, 0.7871058583259583, -0.1552102416753769, 0.06606364250183105, 0.36290550231933594, -2.4915547370910645, 1.1889472007751465, 0.41550102829933167, 0.0516972616314888, -0.2986047863960266, -1.2622102499008179, 1.2752939462661743, 1.543841004371643, -1.4081082344055176, -0.7393433451652527, 1.054274082183838, 1.380207896232605, 0.8900266885757446, -0.14136992394924164, 0.7788311243057251, 0.15685279667377472, -0.09025103598833084, -1.8325049877166748, 1.1717692613601685, -0.19442012906074524, -0.4148321747779846, -0.5875983834266663, -0.7485643029212952, 0.5432343482971191, -0.15685012936592102, 0.2008318156003952, 1.625452995300293, 0.41704681515693665, -0.03824243322014809, 0.20214562118053436, -0.012355625629425049, 0.08200586587190628, -1.0855894088745117, -0.9924237132072449, 0.4399968981742859, -0.16554531455039978, -0.11245260387659073, -0.1516929715871811, 0.12704533338546753, -1.0246086120605469, -0.47973722219467163, -0.7858482599258423, -0.9414879083633423, -0.5757824182510376, -1.0665802955627441, 0.7030316591262817, -0.44522541761398315, 0.8162985444068909, -1.3571962118148804, 0.22404201328754425, -0.8127298951148987, -0.6677735447883606, 0.3368813395500183, 1.2194867134094238, 0.0503915399312973, 0.9009854197502136, -0.28052040934562683, -0.3042226731777191, 0.5620062947273254, -0.45209217071533203, -1.7544803619384766, -1.0308970212936401, 1.022589921951294, -0.6110605597496033, -1.286183476448059, -1.0846619606018066, -0.599195122718811, 0.39909711480140686, -0.9271151423454285, 0.7177007794380188, -1.7504428625106812, -0.41470685601234436, -0.43181514739990234, 0.9198112487792969, 0.47951164841651917, -1.6846182346343994, 0.33916956186294556, -2.0737271308898926, 0.4694471061229706, -0.016862653195858, 0.16854703426361084, 0.9392321705818176, 1.3301347494125366, -1.200796127319336, 0.4455963373184204, 0.6643391251564026, -0.4585247337818146, -1.1395477056503296, 1.119838833808899, -0.7780717611312866, -1.2831426858901978, -0.5371018052101135, 0.9717121124267578, -0.15555183589458466, -0.0017041291575878859, -0.16895964741706848, -0.5501037836074829, 1.2257988452911377, -0.18424592912197113, 0.19292013347148895, 2.0630831718444824, -1.4881433248519897, 0.1509070098400116, -0.6963906288146973, -0.5604265332221985, 1.0452284812927246, 0.3086778223514557, -1.8315002918243408, -0.9797647595405579, 0.4199667274951935, 1.048964500427246, 0.7649538516998291, -0.21304580569267273, 0.6941002011299133, -0.4266735017299652, -0.04776760935783386, -0.737156867980957, 0.13169848918914795, 0.7766242623329163, 1.7235805988311768, 0.36067017912864685, 1.059880018234253, -0.009662632830440998, -0.361833781003952, -1.1575990915298462, 1.055017352104187, 1.8639283180236816, 1.8408597707748413, -1.1478430032730103, -2.335092544555664, -0.6219238638877869, 1.0101025104522705, -0.12823735177516937, 1.2753874063491821, -0.45467036962509155, 1.8155165910720825, 1.145138144493103, -0.5603216290473938, -0.2788321375846863, 0.3422730565071106, 0.09494238346815109, 0.7006515264511108, -0.3773939907550812, -0.10114424675703049, -0.5137935876846313, -0.364103764295578, 1.5858309268951416, -0.5880865454673767, -0.6039212942123413, 1.5311660766601562, -1.7056409120559692, 0.5043032765388489, 0.7195510268211365, -0.06108572334051132, 1.4393303394317627, -1.0109938383102417, -1.6476716995239258, -1.6152100563049316, -0.7546461820602417, -0.37166187167167664, 2.4337806701660156, 0.4665476381778717, -0.9195358157157898, -0.451656311750412, 1.6691986322402954, 4.651069164276123, -0.3118174374103546, 0.6388835906982422, -0.6103274822235107, -0.3699813485145569, -1.3411351442337036, 0.9893370866775513, -0.4013320207595825, -1.1146867275238037, 0.31749099493026733, -0.25118616223335266, -1.6910110712051392, 0.08310822397470474, -0.03482374921441078, 0.12415818870067596, -0.20036952197551727, -0.46347731351852417, 0.3069499135017395, -2.418440341949463, 0.12499567121267319, 1.381248950958252, 1.2246798276901245, 1.2926386594772339, 0.543266773223877, -0.017244134098291397, -0.6054450273513794, -0.14035475254058838, 1.224130630493164, 0.13127179443836212, 0.9101100564002991, -1.9144744873046875, -1.5110511779785156, -0.5782268047332764, -0.6086524128913879, -0.017689036205410957, 0.6487051844596863, -0.08816008269786835, -1.1527308225631714, 0.8839893937110901, -1.1188632249832153, 0.36431512236595154, 0.6580144762992859, 0.11782904714345932, -0.3767525255680084, 1.1273443698883057, 0.08667547255754471, 0.19223380088806152, -1.9310137033462524, 0.16703689098358154, 0.6084617972373962, 0.7751494646072388, -0.19309905171394348, -1.0584739446640015, 0.894773542881012, -2.153900384902954, 0.09504286199808121, -1.3772977590560913, 1.975106120109558, -0.43901199102401733, 0.5133249163627625, 0.2702430188655853, -0.790506899356842, 1.0062196254730225, 1.20442533493042, -2.945871353149414, -1.505324363708496, -0.37980955839157104, -1.3403005599975586, 0.8084072470664978, 0.5759871006011963, 0.23338142037391663, -0.23210321366786957, 1.9366092681884766, -0.21023425459861755, 0.26390448212623596, 0.17827174067497253, 0.09158671647310257, -0.6080906987190247, -0.35723018646240234, 1.5624088048934937, -0.5821475982666016, 0.04841013625264168, 0.29190415143966675, -2.159986972808838, -0.3614366948604584, -0.7915226221084595, -0.8311755061149597, -1.2336267232894897, -0.10359524935483932, -0.06590843945741653, 0.24005863070487976, 0.6398812532424927, 0.39209461212158203, -0.6242557764053345, -1.318705439567566, -0.2761765718460083, 0.06039268523454666, 0.9739210605621338, 0.626039981842041, 1.1763306856155396, -0.37658023834228516, -0.25871583819389343, -0.5446698069572449, 0.15740351378917694, -1.3475910425186157, -0.4481041431427002, 0.7014769315719604, 1.33700692653656, 0.31835752725601196, 1.3823920488357544, -0.8635137677192688, 0.5312292575836182, -0.5136721730232239, 0.534832775592804, 1.3456767797470093, 0.18559832870960236, -0.4034007787704468, 0.568448543548584, 0.10303985327482224, -0.2164396345615387, 0.055304065346717834, -1.7664614915847778, -0.9499167203903198, -2.894930124282837, -2.1424810886383057, 1.4015947580337524, 0.9606927633285522, 1.1609370708465576, -0.5474562644958496, -0.2000495195388794, 1.046063780784607, 0.8856045007705688, -0.21767625212669373, -1.6655607223510742, 0.8634359836578369, 0.6457934379577637, -0.7125847339630127, -1.4591670036315918, 0.4831744432449341, -3.080320358276367, -0.6713720560073853, -1.4230530261993408, -0.6078499555587769, 0.35474351048469543, -0.4850018620491028, 0.5517183542251587, 0.47494426369667053, 1.9088541269302368, 0.6854381561279297, 0.21990060806274414, 0.16468024253845215, -0.05264335125684738, -2.1403138637542725, 0.3071787357330322, -1.1108964681625366, 0.07828985899686813, -1.3412189483642578, 1.561314582824707, -1.1314871311187744, -2.196442127227783, 0.4524361491203308, 1.3041859865188599, -0.9997006058692932, -1.362382411956787, -1.6114097833633423, -0.6284828186035156, -0.6906551122665405, 2.062222957611084, 0.19394099712371826, -1.6806005239486694, -1.3665281534194946, -0.8180299997329712, 0.23111926019191742, 1.2569423913955688, -1.739062786102295, -0.28674525022506714, 0.024492967873811722, -1.4059034585952759, 1.1522111892700195, 0.31300824880599976, 1.281091570854187, -1.4478174448013306, -0.40134039521217346, 0.5639368295669556, -0.7573421001434326, 0.5662893056869507, -0.7399848103523254, -1.7548245191574097, 1.297473430633545, 1.1677637100219727, -0.19163472950458527, -1.3758426904678345, 0.09182144701480865, 0.32051780819892883, -0.6010781526565552, 0.504044234752655, 1.8215833902359009, 0.770167887210846, -0.9266868233680725, -0.06931369006633759, 0.732384204864502, 1.1293190717697144, -0.18005786836147308, -1.2818899154663086, -0.7900272607803345, -0.527758002281189, 0.24020913243293762, 0.07147307693958282, -2.4657976627349854, -1.2020293474197388, 0.8843783736228943, -1.661616325378418, -0.059039175510406494, -0.454378604888916, -2.516265869140625, 0.7049928903579712, -1.105576515197754, 1.6052173376083374, -0.010907319374382496, 1.2708193063735962, 0.7729750871658325, 0.3378347158432007, -0.9238836169242859, 1.3592965602874756, 0.3556132912635803, -1.0600261688232422, -0.7745791077613831, -2.534597396850586, -0.969273030757904, 0.5835081338882446, 0.6409783959388733, -0.027517501264810562, 0.9205824136734009, -3.304975748062134, -2.796265125274658, 0.5326268076896667, -0.9006586670875549, 0.4762950539588928, -1.2194159030914307, 0.3675163984298706, -0.8496845364570618, 0.19858546555042267, 0.4698348343372345, 0.6011019945144653, -1.3320858478546143, -0.699722945690155, -0.898027241230011, 0.03337778523564339, 0.4961288571357727, -0.9623728394508362, 1.6032118797302246, -0.04990725591778755, 1.6965988874435425, 1.2112280130386353, -0.9899281859397888, -0.42472636699676514, -0.7060948014259338, -0.8153735399246216, -0.06903249770402908, -0.8399719595909119, 0.9831746816635132, -0.3489828109741211, -0.23188146948814392, -0.41756966710090637, 0.3252946734428406, -0.016931233927607536, 0.12130405008792877, -0.3772025406360626, 0.47718942165374756, -0.5812565088272095, -0.6510184407234192, -1.0971481800079346, 0.6813458800315857, 1.564487338066101, -0.2528720200061798, -0.13304583728313446, 1.8171052932739258, 0.2852857708930969, -0.370877206325531, -0.8811615109443665, 1.8333187103271484, -1.9043906927108765, 1.0072139501571655, 0.5208648443222046, -0.607783317565918, 0.2630743980407715, 1.7318843603134155, 0.2812117636203766, -0.09801395982503891, 0.2782125473022461, -0.40581953525543213, 3.396047592163086, -1.6843398809432983, 1.4876019954681396, -0.3234076499938965, -0.47500234842300415, 0.9947129487991333, 0.9563310146331787, -2.3081130981445312, 1.275102138519287, -1.2073156833648682, 0.6840776801109314, 0.18423578143119812, 0.7704105973243713, -0.6289097666740417, 1.14371919631958, 0.4047141671180725, 0.06946622580289841, -0.8355346918106079, 0.1621214747428894, -0.7690160870552063, -0.35341712832450867, -1.5532002449035645, -0.03723914548754692, -0.5299652814865112, -0.6790177822113037, -0.7440907955169678, 1.7074894905090332, 0.2764357626438141, 0.6793650388717651, 0.9214349985122681, 0.6973028182983398, -0.5039740800857544, 0.23090320825576782, 0.07353498041629791, -0.3492276668548584, -0.748688280582428, -2.5113346576690674, -1.4608482122421265, 0.8360939621925354, 0.9555601477622986, 0.17191994190216064, 1.60130774974823, 0.9913570284843445, -0.4866198003292084, -0.6714668869972229, -0.462704062461853, -1.2324950695037842, 0.75129634141922, 0.34979236125946045, -0.20838724076747894, 0.6783194541931152, -1.4385693073272705, -1.1863635778427124, 0.614872932434082, 0.38064703345298767, -0.6862051486968994, 2.1653480529785156, 0.593605101108551, 0.40680426359176636, -0.19998763501644135, 0.6363441348075867, -0.5058130025863647, 0.2892022430896759, 0.925034761428833, -0.39325451850891113, 1.739196538925171, -0.9709767699241638, 0.3111027181148529, 0.8839358687400818, -2.0784077644348145, -1.010514259338379, -0.5669485926628113, 0.42560824751853943, -0.7991937398910522, -1.3075532913208008, -0.5186439156532288, -0.5520438551902771, -1.1156103610992432, 0.5693091154098511, -0.667793333530426, 0.7947537302970886, -0.4769837558269501, 1.7860056161880493, 0.8110063672065735, 0.3444220721721649, -0.1580527126789093, 0.5940247178077698, -0.3585227131843567, 0.9958345890045166, 0.5732113122940063, -1.0242739915847778, -0.049664001911878586, -0.3549298346042633, 2.0174400806427, -0.49348998069763184, 2.5542619228363037, -1.1234171390533447, -1.6868313550949097, -0.268685907125473, -0.09533992409706116, -1.5695056915283203, -0.22409404814243317, -0.0801331177353859, -0.4741978049278259, -1.6456029415130615, -0.585830569267273, 1.2932312488555908, -0.8860505223274231, 0.1040256917476654, 0.04690611734986305, 0.5795015096664429, 1.728118896484375, -0.14199107885360718, -0.5563983917236328, -0.9151535034179688, 0.5430567264556885, 0.48508256673812866, -0.4723367989063263, 0.8245962858200073, -1.8917821645736694, 0.19687463343143463, -0.2568235993385315, -0.19142499566078186, -0.28370150923728943, -0.9247938990592957, 0.42846137285232544, 0.41849279403686523, 0.9607946276664734, -1.6852813959121704, 0.7472926378250122, -1.6082314252853394, 0.0004737918497994542, 0.9563765525817871, 0.8231227397918701, -0.049064576625823975, 0.24332936108112335, -1.1638864278793335, 0.1485283374786377, 0.2320639044046402, 0.5457573533058167, -0.4222443401813507, -0.962569534778595, -1.2101514339447021, -1.1374720335006714, 0.3238903284072876, 0.2956138849258423, -0.02522612363100052, 2.425691604614258, 0.9679487347602844, -0.4688003659248352, -0.05591661483049393, 0.49761611223220825, -1.1495516300201416, -1.056972861289978, -0.21828149259090424, -0.5937674045562744, -2.1354856491088867, -0.7692162990570068, -1.1812280416488647, 1.3693428039550781, 0.5166410803794861, 1.1782582998275757, -1.092704176902771, -0.33431679010391235, -0.642919659614563, -2.2594730854034424, -0.5282912850379944, 2.2113864421844482, 0.2276046872138977, -1.1632819175720215, 0.4316593110561371, -0.7599166035652161, 0.8946097493171692, 2.078948974609375, 0.684450626373291, 0.06737203896045685, 1.6918493509292603, 1.6263853311538696, -0.7399120926856995, 0.9553844332695007, 2.4663522243499756, -1.58359956741333, -0.47541263699531555, 0.8686304688453674, 1.9251328706741333, 0.5239792466163635, 0.6937344074249268, 0.47159287333488464, 1.5102639198303223, -1.8125876188278198, -2.146624803543091, -0.6668685674667358, -2.8230233192443848, 0.9644976854324341, -2.124044179916382, -0.17470595240592957, -0.9258919358253479, 0.8952220678329468, 2.508740186691284, -0.9508737325668335, 0.09547742456197739, 1.3640908002853394, -0.558163583278656, -0.4990215003490448, 0.9409606456756592, 0.9748662114143372, 1.197785496711731, 0.004516801796853542, 1.237100601196289, 0.8217021822929382, 1.1144747734069824, -1.0115422010421753, 0.08909675478935242, -0.7338216304779053, 0.7167056798934937, 0.10672180354595184, -1.0088146924972534, 1.1267499923706055, -0.9901176691055298, 1.6944849491119385, 0.17808374762535095, -0.6711024641990662, 2.173866033554077, -1.7346192598342896, -0.8799654245376587, 0.07011040300130844, 1.4522526264190674, -0.4451459050178528, -0.5327578783035278, -0.5866685509681702, 1.1177146434783936, 0.2984664738178253, 0.7208696603775024, 1.885329008102417, 0.004242132417857647, 0.7225687503814697, -1.8333699703216553, 0.3829362094402313, 0.5768405199050903, -0.08485747128725052, -0.6253318190574646, 0.5582271814346313, 1.6039100885391235, -0.21772229671478271, -0.9402657747268677, 0.31457602977752686, 0.5834870934486389, 0.37396296858787537, -0.726093053817749, 0.5628554821014404, 0.10995295643806458, 1.2036471366882324, -0.8874420523643494, -0.28319111466407776, 0.3021487593650818, 0.10927728563547134, -0.4642455577850342, -0.41281914710998535, -1.2392204999923706, 1.525517463684082, 0.7747636437416077, 0.4735622704029083, -1.1907227039337158, -1.291256308555603, -1.1185165643692017, 0.8533260226249695, 0.45736196637153625, -1.1304726600646973, 0.9791776537895203, 0.9746783375740051, -0.45144134759902954, 1.2762359380722046, -1.3269541263580322, -1.5504697561264038, -2.052558660507202, -1.112913727760315, -1.310904622077942, -0.912753701210022, -0.5007064938545227, -0.24334192276000977, -1.0867769718170166, 1.6076841354370117, -0.41897013783454895, -1.406795859336853, 1.7833092212677002, -2.078080177307129, 1.5244603157043457, 0.13726016879081726, 1.0817387104034424, -0.3380948603153229, -0.9606146216392517, -0.7346422076225281, -0.11990425735712051, -1.8121471405029297, -2.3447914123535156, -0.9893022179603577, 0.211884006857872, 0.049188122153282166, 2.6440136432647705, -1.3091899156570435, 0.022973787039518356, -0.40537863969802856, -2.2933781147003174, -1.5412790775299072, 0.2752131521701813, 0.7266733050346375, -0.6199307441711426, 0.27228641510009766, -0.6627771854400635, 0.0032604397274553776, 0.045045316219329834, -1.1176490783691406, 0.3230508863925934, 0.8772711753845215, 0.36418890953063965, -0.8617345094680786, 1.3237048387527466, 0.6032006144523621, 0.6901139616966248, -1.724340558052063, 0.23086971044540405, 0.5902188420295715, 0.10929574817419052, -0.5062623620033264, 0.9169224500656128, 0.4974799156188965, -0.42390701174736023, 0.1617172211408615, -0.5377612113952637, 0.36359742283821106, -0.4453885853290558, -0.5879238247871399, -0.3896433711051941, -0.6178739070892334, -1.2588376998901367, -0.11320353299379349, 0.28470829129219055, 0.8587048649787903, -2.055310010910034, -0.2547377943992615, 0.6970875263214111, -0.2581884264945984, -0.5878081917762756, 0.03878391534090042, 0.3855454921722412, 0.3568216860294342, -1.2933564186096191, 0.8833039999008179, 1.1645582914352417, -1.092818021774292, -0.3183966875076294, -2.04350209236145, 0.277101993560791, 1.2433435916900635, -0.17148169875144958, 0.8209624290466309, 0.3661939203739166, 0.6432909965515137, 1.302150011062622, -0.6028769016265869, 0.6353587508201599, -0.40194499492645264, -0.19550465047359467, 0.5681341290473938, 0.1137581318616867, 0.28434211015701294, -0.3439568281173706, 0.42469415068626404, 0.7492988109588623, -0.9145355224609375, 0.17019639909267426, -0.007586701773107052, 1.6672359704971313, -1.7003886699676514, 0.7684296369552612, -1.5742201805114746, 0.16302531957626343, 0.4415133595466614, -0.7384192943572998, -0.9243489503860474, -2.4200375080108643, 0.9074276685714722, -1.1163582801818848, 0.3704579472541809, 0.37271857261657715], "dtype": "float32"}, {"__nd__": [-0.6439723372459412, 0.6645553112030029, -0.008813640102744102, -0.9780003428459167, -0.4655674397945404, -1.0482630729675293, 1.6366955041885376, 0.6893052458763123, 0.01509647723287344, -1.8276913166046143, 0.6173536777496338, -2.1381280422210693, 0.4176986813545227, 1.4809051752090454, -0.032142333686351776, 1.5778498649597168, -0.7491554617881775, 0.6313453316688538, 0.09882977604866028, 1.1169743537902832, -1.1627142429351807, -0.09722661226987839, -0.5421470403671265, -0.7281815409660339, -0.6673785448074341, -0.9268171787261963, -1.7276966571807861, 0.14101336896419525, 1.083796501159668, -0.09623300284147263, 0.41531968116760254, 0.39653220772743225, -0.7340995669364929, 0.196241557598114, 0.11450877040624619, -0.9563588500022888, -0.2757270038127899, 0.6029573678970337, -1.1742173433303833, -0.15235170722007751, 1.6765154600143433, 1.6926910877227783, 0.3011329472064972, 0.7837661504745483, -0.23502105474472046, -0.0022385502234101295, 0.5399110913276672, 1.0791634321212769, -1.099847674369812, -1.366455316543579, -0.9446173310279846, 1.0850998163223267, 0.1308109164237976, 1.2701196670532227, 0.4440217614173889, 0.6739119291305542, 0.6070371270179749, -0.028178690001368523, 0.08303869515657425, -0.3508709669113159, 1.3390183448791504, -0.6701802015304565, -1.0141551494598389, 1.4547405242919922, -1.054231882095337, 1.413551926612854, -0.0895061194896698, -1.075236201286316, -0.5081477165222168, 0.6628588438034058, 0.796442449092865, 0.8237485885620117, -1.0771522521972656, 0.6192190051078796, 0.8517071008682251, 0.5365548729896545, -0.8625165224075317, 0.8765904903411865, 1.0557847023010254, -0.10039568692445755, -0.8202605843544006, 0.23352359235286713, -1.927949070930481, 1.0817826986312866, 0.24108512699604034, 1.355760097503662, 0.8259887099266052, -1.5404266119003296, -0.1263544112443924, 1.6641175746917725, 0.544757604598999, -0.9413557052612305, 1.3701956272125244, -0.12723733484745026, -1.3183603286743164, 0.5808537602424622, -0.009998384863138199, -0.7219321727752686, 0.09901208430528641, -0.18779368698596954, 0.07166175544261932, -0.7346084713935852, 0.37346145510673523, -1.7613952159881592, -0.29139238595962524, -0.4488450288772583, -0.8528096675872803, 0.08563882857561111, 0.23774714767932892, -1.0045826435089111, 2.203289031982422, 0.11399820446968079, -0.269158273935318, 0.27451255917549133, -0.009696087799966335, 0.45394328236579895, 0.6018556356430054, 0.5971082448959351, -1.9420013427734375, -0.3564230799674988, -0.7139672636985779, -0.4183657467365265, -0.9641528129577637, -1.6266928911209106, 0.9314643144607544, -0.9548574090003967, 1.8341984748840332, 0.2011546790599823, 0.060310471802949905, 2.0288944244384766, -0.7874787449836731, 0.6013613343238831, 1.15486478805542, -0.09968894720077515, 0.5537428259849548, 0.49673816561698914, 1.7857694625854492, -0.9415093660354614, 0.5072248578071594, -1.317131519317627, 0.4910302460193634, -0.7713520526885986, 1.1797456741333008, 0.3368869721889496, 1.1872000694274902, 1.3384146690368652, -0.06867965310811996, -1.4239850044250488, -1.6046675443649292, -1.8524677753448486, -0.6380206346511841, -1.5175150632858276, 0.3408661484718323, 0.8094097971916199, 2.0447027683258057, -1.6020509004592896, 1.1753582954406738, -0.704333484172821, -0.00881109293550253, -1.894594669342041, 0.43111222982406616, 0.6351091861724854, 0.10872718691825867, 0.4516792595386505, 0.3219751715660095, 0.4882841110229492, 0.007706772070378065, -0.33996227383613586, 1.1174304485321045, 0.8318058252334595, 1.8388876914978027, 0.8500375151634216, 1.518183946609497, -0.22589020431041718, -0.4854290187358856, 0.6238564848899841, 0.8617222309112549, -0.12823453545570374, 0.128370463848114, -1.5996791124343872, 0.62655109167099, 2.187498092651367, 2.7209932804107666, 0.2792724668979645, -0.49778226017951965, -0.5566187500953674, -0.368561714887619, 0.4382001459598541, 0.8325289487838745, -0.37306177616119385, -0.38264405727386475, -1.2421931028366089, 3.100220203399658, -0.9995642900466919, -0.086609847843647, 1.2699249982833862, 1.1058682203292847, -0.05109955370426178, -1.1998018026351929, 1.4805983304977417, 1.0574977397918701, 0.623387336730957, 0.8694155216217041, -0.9082730412483215, -0.625496506690979, 0.5290627479553223, 0.6065019369125366, -0.4509821832180023, 0.2598928213119507, -1.3071726560592651, -0.26430198550224304, -0.4137985110282898, 1.234575867652893, -0.968554675579071, -0.6528393030166626, -1.1621425151824951, 0.17229843139648438, -0.1530989557504654, -0.8726963400840759, 0.15125110745429993, -0.9786258339881897, 1.7828289270401, 0.6568489670753479, 1.0899523496627808, 1.7754170894622803, -1.4854637384414673, -1.163527011871338, 0.5417316555976868, 0.976466178894043, -0.44628140330314636, 0.8368052244186401, 0.9646669030189514, -0.8687003254890442, -1.2877471446990967, 0.5685562491416931, 2.055168867111206, 0.24917800724506378, -0.9085224270820618, 1.5862760543823242, 0.36901137232780457, -0.731053352355957, -0.13018916547298431, 0.995294988155365, -1.811980128288269, 1.7449920177459717, 1.06111478805542, -0.13306638598442078, -0.05305487662553787, 0.6300352215766907, -1.1455020904541016, 1.0288076400756836, -0.13281525671482086, -0.17808665335178375, 1.251830816268921, 0.8708119988441467, -0.6681272983551025, -2.18807315826416, 0.4839417040348053, 0.14143401384353638, 0.8076171875, 0.5210388898849487, 0.028835661709308624, 0.567581057548523, -0.2876725196838379, 0.6005522608757019, -0.39266443252563477, -0.3488326668739319, 0.9311792850494385, 0.46834683418273926, -0.6673474907875061, 0.0009848036570474505, -0.5780162215232849, 0.0756133496761322, -2.0711450576782227, -0.35712119936943054, 0.42584627866744995, -0.055015597492456436, 0.053995195776224136, -1.4205646514892578, -1.9075332880020142, -1.6356042623519897, 0.4308296740055084, 0.4911963641643524, -0.6710743308067322, 0.5517081618309021, -0.7591404914855957, 0.7631836533546448, -1.0597503185272217, 0.500419020652771, -0.11828144639730453, 0.16938090324401855, 0.7109041810035706, 0.3122159242630005, 1.2329365015029907, 0.18010179698467255, 1.838168740272522, -0.12154057621955872, 1.021007776260376, -0.09000767767429352, 0.07557227462530136, -0.5966772437095642, 0.7135298252105713, -0.04952695593237877, 1.8234355449676514, 1.0385688543319702, 1.2216527462005615, -1.7142335176467896, 1.455242395401001, 1.206779956817627, -1.9148473739624023, -0.213955819606781, -0.9731413125991821, -0.25142887234687805, 1.7286245822906494, 1.4523648023605347, 0.4415183961391449, 0.041672080755233765, 0.7953284978866577, 1.6374534368515015, 0.1760762482881546, -0.6241720914840698, 0.38543468713760376, -1.3024816513061523, 0.11059020459651947, 0.8391911387443542, -0.367549866437912, -0.9109734892845154, 1.0833600759506226, -0.011468102224171162, -0.2835911214351654, 1.0172381401062012, -0.3610279858112335, 1.5890388488769531, -1.8079044818878174, -0.10690333694219589, 0.49540483951568604, 0.38442254066467285, 2.16518497467041, 0.15082094073295593, 0.6858294606208801, 0.7729299664497375, -0.578298032283783, 1.2901694774627686, 0.8027230501174927, -0.5935561656951904, -0.3520582616329193, -0.4016091823577881, 0.22359658777713776, 2.103663921356201, 1.1084232330322266, -0.8182090520858765, 0.586151659488678, 0.23462346196174622, -0.5325592756271362, 0.5018439292907715, 0.7166211605072021, -1.2101343870162964, 2.579639434814453, -1.0711919069290161, -0.7680879831314087, -1.2113853693008423, 0.3058778941631317, -0.2618142366409302, -0.4663848876953125, -1.334619164466858, -0.5910388827323914, 0.3985992670059204, -0.4180524945259094, -2.2971031665802, 0.044515203684568405, -1.4285633563995361, 1.9294370412826538, 0.25002068281173706, -0.41410088539123535, -0.9524720907211304, 1.4858540296554565, 0.28715381026268005, 0.14363670349121094, -1.2596429586410522, 1.8793630599975586, -0.9305621385574341, 0.41333410143852234, -1.570190191268921, -0.09294244647026062, -0.7069936394691467, 2.3726587295532227, -1.1743249893188477, 0.9063417315483093, 1.7909942865371704, 0.7274667620658875, -0.24913309514522552, 1.486196756362915, 1.0320312976837158, 0.2595069408416748, -1.3438597917556763, 1.3922194242477417, 0.7748289704322815, 2.801591396331787, 0.8388809561729431, 2.8651509284973145, 0.7719123959541321, 0.3611563444137573, 0.7599769234657288, 0.6802861094474792, 0.22490116953849792, -0.9533860087394714, -0.08320922404527664, 1.220101237297058, -2.138129234313965, -0.8616064786911011, 0.24035605788230896, 0.8011261820793152, 0.673869788646698, 2.197005033493042, 1.6472077369689941, 0.02701646462082863, 0.24734386801719666, 0.22983868420124054, -1.2140638828277588, 0.6567743420600891, 0.18931794166564941, -1.9962785243988037, -0.8881489038467407, -1.3727078437805176, 0.30660074949264526, 0.12902218103408813, 1.5472917556762695, 0.46838945150375366, -0.10989049077033997, -0.10282205790281296, 0.6022563576698303, 1.1051154136657715, -1.2659374475479126, 1.0808507204055786, 1.3433071374893188, 0.10181993246078491, -0.1262190192937851, -0.9174599051475525, -0.9455684423446655, -0.6836539506912231, 0.3395218849182129, 1.5313526391983032, 0.19161102175712585, -0.8532684445381165, 1.1584571599960327, -0.30614081025123596, 0.10642807185649872, 0.8215265274047852, -0.3841795325279236, -1.2754700183868408, 0.18679068982601166, 0.5115335583686829, -1.7771435976028442, -0.3487323224544525, -0.020029984414577484, -0.3366284966468811, 2.218803882598877, -0.7944348454475403, -0.8290605545043945, -0.7780494093894958, -0.2033696174621582, 2.032066822052002, 2.206731081008911, -0.21705090999603271, 1.0900036096572876, -2.9718987941741943, -0.30354607105255127, -0.24329409003257751, -0.04901096969842911, -0.3595937490463257, 1.7225390672683716, 1.065567135810852, 0.28995954990386963, -0.14447839558124542, -1.1247494220733643, -0.055151503533124924, 0.6986863613128662, -0.20299465954303741, -2.333155393600464, 0.5383530259132385, -0.6860787868499756, -0.8707743883132935, -0.987803041934967, -0.21726348996162415, 0.49029719829559326, 1.7395777702331543, 0.2702258825302124, -0.6559171676635742, -0.00040337248356081545, -0.7394071817398071, -0.4811653792858124, -1.4476007223129272, 0.1265038549900055, 1.052272081375122, -1.0984498262405396, 1.3060327768325806, 1.3682072162628174, -0.49578657746315, 1.00527822971344, -1.2759037017822266, -1.9400092363357544, 0.7776690721511841, 0.1947057545185089, 0.036040984094142914, -0.7167918682098389, 0.5838966965675354, -1.1027846336364746, -1.5745543241500854, 0.21145759522914886, -0.05902201309800148, 0.35072553157806396, -1.8232321739196777, 0.40920376777648926, 0.5777032375335693, -0.3816026449203491, -0.40937501192092896, 0.5943809747695923, 0.5580418705940247, 0.5149390697479248, -0.741791844367981, -0.7738807201385498, 0.9169936776161194, -0.26157301664352417, -0.5357582569122314, 0.22873064875602722, 1.4068325757980347, -0.1585080921649933, -0.978280782699585, -0.669302761554718, 0.05414684861898422, 0.7966843247413635, 1.7275086641311646, -0.30915385484695435, 1.010717511177063, -0.6508520841598511, 1.6304495334625244, 0.46072593331336975, 1.6907931566238403, 1.147122859954834, 0.07955712080001831, 0.5303294658660889, 0.5951350331306458, 0.25463956594467163, -0.6067832112312317, -0.548215389251709, 0.625140905380249, -1.2632392644882202, -1.029425024986267, 0.07692258805036545, -0.9498706459999084, -1.2783253192901611, 2.551945447921753, -2.528416156768799, -1.2172828912734985, -0.792317271232605, 0.8560640215873718, -0.47448623180389404, -0.45125842094421387, 0.09839124232530594, -0.23965100944042206, -1.6379070281982422, 0.33567285537719727, -1.3679858446121216, -1.5369653701782227, -0.2455148696899414, 0.6441855430603027, 1.5655014514923096, 0.5707516670227051, -0.3296659588813782, -1.2156966924667358, 0.11952707916498184, 0.14789557456970215, 0.9521823525428772, 0.5801857113838196, 0.32473334670066833, 1.0928040742874146, -1.5657365322113037, 0.3342413902282715, -1.087040901184082, -1.600327968597412, -1.2259434461593628, 0.4740629196166992, 0.6542325615882874, 0.6317858099937439, 0.1593088060617447, -0.38659483194351196, 0.3802872896194458, 0.18578948080539703, 0.11683517694473267, 0.23985479772090912, 0.12380784749984741, 0.20303793251514435, -0.8688916563987732, 0.4758690595626831, 1.7853779792785645, 0.3013034760951996, -0.29277464747428894, -1.4557762145996094, 0.5922651886940002, 0.7346587777137756, -0.305496484041214, 1.3043367862701416, 1.5365954637527466, -0.6067137122154236, 0.5046781301498413, 1.5073215961456299, 0.16671185195446014, -0.7686333656311035, -0.5902698040008545, -0.8260992169380188, 0.1827535480260849, 0.8176848292350769, 0.8820529580116272, 1.6438006162643433, -0.6790199875831604, 0.7091106176376343, 1.9650055170059204, -2.5105574131011963, 0.019670184701681137, -0.22607292234897614, 0.7773381471633911, -0.6798380613327026, 1.1601051092147827, 0.0367552675306797, -0.044198352843523026, 0.7301334142684937, -1.2013872861862183, -1.1542856693267822, 1.8810131549835205, -0.3186713755130768, 0.8405262231826782, 1.3168846368789673, -2.0034825801849365, -1.9681017398834229, 1.994792103767395, 1.9457213878631592, -0.9113051295280457, -0.14830325543880463, 1.244201898574829, 1.734037160873413, -1.2723493576049805, 0.8545839190483093, 0.6545501947402954, 0.9402599334716797, 1.4425665140151978, 0.532374382019043, 0.18084225058555603, -0.31884095072746277, 0.15435323119163513, 0.3805956244468689, 1.124015212059021, -0.1254134476184845, -1.1154234409332275, -0.8907091617584229, -0.971172571182251, -0.011130988597869873, 0.5871102213859558, 0.09041747450828552, 0.019123876467347145, 0.018305040895938873, 1.4400798082351685, 1.1551125049591064, -0.7447764277458191, -0.625566840171814, -0.20684784650802612, 1.3768810033798218, -0.42254045605659485, -0.6069944500923157, 0.6385766267776489, 0.5357142090797424, -0.08222179114818573, 0.6732994914054871, -1.277968168258667, 0.42632439732551575, -0.9697835445404053, 0.7303199172019958, 1.2483618259429932, -1.2571214437484741, 0.922839879989624, 0.9162298440933228, -1.158853530883789, 0.4396875500679016, -0.5874531865119934, -1.1323968172073364, 1.4206572771072388, 1.0568312406539917, 0.36593735218048096, -2.097054958343506, -1.1900355815887451, 2.1516661643981934, -0.882012665271759, 0.7697746753692627, -1.2708244323730469, 2.6243479251861572, 1.1069343090057373, -1.6120719909667969, 0.5946285724639893, -0.4123004972934723, 0.12644851207733154, -0.806536078453064, 1.254345417022705, 0.6562029123306274, 0.21247874200344086, 0.8173536062240601, -1.0848193168640137, 0.021084481850266457, 0.630650520324707, 1.3516807556152344, -0.2734760642051697, 0.6526615023612976, 0.4664418399333954, -0.9456202983856201, 0.23319634795188904, 0.34575599431991577, 1.3675572872161865, 0.9093074798583984, 0.6304269433021545, -0.9914590716362, 0.5336945652961731, 0.7506710886955261, 0.271102637052536, -1.1872659921646118, -0.061908211559057236, -0.44445401430130005, -0.3659302890300751, 1.6129075288772583, 0.04485740140080452, -0.7276807427406311, 0.3276607394218445, 0.3685672879219055, 0.14137084782123566, -0.8809451460838318, -0.5663954615592957, 0.25818610191345215, -1.1428799629211426, -0.6180806159973145, -0.8174057602882385, -0.7732701301574707, -0.7780503630638123, 0.21370287239551544, 0.4068908393383026, -0.22246579825878143, -1.3012504577636719, -0.6226950883865356, -0.032975852489471436, -0.35191306471824646, 0.15798453986644745, -0.4531664550304413, 0.8704137802124023, 0.4830966889858246, -0.7151765823364258, 1.6787569522857666, -0.6909217834472656, 0.02393934316933155, -0.24406807124614716, -0.07073880732059479, -0.4953640401363373, -0.0384061373770237, -0.22275637090206146, 0.07969437539577484, -0.5853736996650696, -0.6324301958084106, -0.8546581268310547, -0.7255145907402039, -0.3199846148490906, -1.5308949947357178, -0.5864119529724121, -0.8867968320846558, -1.0869784355163574, -0.35558971762657166, -1.9765141010284424, -1.6646212339401245, 3.017637014389038, 0.7780312299728394, -0.1041242852807045, 1.1176701784133911, 0.5684983730316162, 1.2222990989685059, -0.014344385825097561, -1.1945537328720093, 0.9910230040550232, -1.5442750453948975, 1.0488858222961426, -1.3204600811004639, 2.113755464553833, 0.5273168683052063, -1.026581048965454, -0.4026225805282593, 1.1872276067733765, 0.6001114249229431, -0.8413193225860596, 0.8544026017189026, -0.17548522353172302, -1.0904020071029663, -0.9419590830802917, -1.8529725074768066, -0.42166969180107117, 1.2674639225006104, 0.18051889538764954, 1.6331713199615479, -0.09305816888809204, 0.8475427627563477, 1.3294727802276611, -0.5994827747344971, 1.1262272596359253, 0.08358091861009598, -1.2309261560440063, 0.4582412540912628, 0.16353602707386017, 0.41609466075897217, -1.9789575338363647, 2.0526936054229736, 0.3950009047985077, 0.8607726693153381, -0.40891972184181213, 0.4795260727405548, -0.2516738176345825, 0.43127739429473877, 0.4410991072654724, 0.15014809370040894, -0.10140222311019897, 1.677211880683899, -0.15333333611488342, -0.03266159072518349, -1.0705195665359497, -2.647705316543579, 0.3817949891090393, 0.07421436160802841, -0.3570372462272644, -0.3829237222671509, -0.4504212737083435, 1.1063319444656372, -0.6417287588119507, -0.5241588950157166, 1.6747552156448364, 1.948932409286499, -0.6640998125076294, -0.3425244987010956, 0.16998201608657837, -1.0469118356704712, -0.5935630202293396, -0.13428111374378204, -1.224252700805664, -0.9794744849205017, 0.5869565010070801, 0.1363970935344696, 1.3594073057174683, 1.7830442190170288, 1.0198686122894287, -0.2531672716140747, -0.13930077850818634, 0.6719920635223389, 0.0029404705855995417, -0.27959662675857544, -1.4003586769104004, -0.9621644616127014, 0.7090725302696228, 1.7759106159210205, -0.24575164914131165, -1.0743553638458252, 1.1399849653244019, -1.0700414180755615, -0.26403629779815674, -1.4803721904754639, -1.0432171821594238, -0.09716968983411789, -0.390865296125412, -0.6036470532417297, 0.9633955359458923, 0.790855348110199, 0.7392709851264954, 2.2167978286743164, -0.30452316999435425, 1.264609932899475, 0.33872082829475403, -0.8586781024932861, -0.93471759557724, 1.214071273803711, -0.47523581981658936, 1.4641913175582886, -0.5473297834396362, 1.7356197834014893, -0.8658809661865234, 1.7021249532699585, 1.0889633893966675, -0.6263524889945984, -0.6489582657814026, -1.0316157341003418, 0.3249298930168152, 0.03637594357132912, 0.9713143110275269, 0.14006437361240387, 0.9892306327819824, -0.03129817172884941, -0.8976878523826599, 1.009663462638855, 0.752467930316925, -1.0861084461212158, 1.368897795677185, -0.5476575493812561, 1.103690266609192, 1.8360708951950073, 0.6094914674758911, -0.49240538477897644, 0.11813869327306747, -0.24815672636032104, -1.5033739805221558, 0.5743106603622437, 2.630627155303955, 1.2408931255340576, -0.06348419189453125, -0.08974695950746536, -0.5834317803382874, 0.268985390663147, 0.5823145508766174, -0.8941311836242676, 0.17504183948040009, 0.514043390750885, -0.10709740966558456, -1.4863452911376953, 0.10673724114894867, 0.3778633177280426, -0.4199376404285431, 0.9678006172180176, 0.6694530248641968, -0.755096971988678, -2.326719045639038, 0.18466702103614807, -0.20443645119667053, 1.0713623762130737, 0.11137376725673676, -0.10524041205644608, 1.6385035514831543, 0.4788060486316681, -0.6868715882301331, 0.21911008656024933, 0.3386838436126709, -0.9397549629211426, 0.3332720100879669, 1.0998998880386353, 0.8163353204727173, 0.21430976688861847, -0.6792813539505005, -0.7657850384712219, -0.8065630793571472, 1.6760334968566895, 1.144628882408142, -0.6929315328598022, -2.170440673828125, -0.09717221558094025, -0.3664013743400574, -0.7417675852775574, 0.4570874571800232, 0.42234963178634644, -0.7823966145515442, -0.28604111075401306, 0.2024284303188324, -1.8991955518722534, 0.9260196089744568, 1.4669697284698486, -1.263617753982544, 0.479768306016922, 1.149192452430725, -0.6125267148017883, 0.4529707729816437, -1.2950987815856934, -1.1059683561325073, 0.8673689365386963, 1.5626462697982788, -0.13062553107738495, -0.22065038979053497, 0.2656826376914978, -0.09725689142942429, -0.15891210734844208, 0.49492621421813965, 0.07306836545467377, 0.18835876882076263, 0.21829108893871307, 0.13812609016895294, 0.18669451773166656, 0.4422816038131714, -0.6053104400634766, -1.433529019355774, -1.1053636074066162, -0.0697251409292221, 0.39774376153945923, -0.9452729225158691, 0.23648281395435333, 0.5464217662811279, -0.8743071556091309, 1.076159119606018, 1.1037156581878662, 0.8330399394035339, -1.2596906423568726, -0.03898565098643303, -1.0031027793884277, 0.12619172036647797, -0.6042750477790833, 1.4789788722991943, -1.151740312576294, 0.36931633949279785, -1.0713173151016235, -1.1221455335617065, 0.7607656717300415, -0.011559953913092613, -0.011693904176354408, 0.6119160056114197, -0.38660648465156555, 0.4115717113018036, -0.979873776435852, 1.3905391693115234, -0.6779420375823975, 0.13980460166931152, 0.8027932047843933, 0.40742772817611694, 0.32850831747055054, 1.043878436088562, 0.5817880034446716, -0.16535061597824097, -1.3765567541122437, -0.36280274391174316, -0.6596986651420593, -0.7524864673614502, -0.20927931368350983, 0.05957973748445511, 0.0656440407037735, 0.36531171202659607, 1.061608076095581, -1.3996899127960205, -1.02130126953125, 0.5958305597305298, 1.0623847246170044, 0.008011224679648876, -0.022437138482928276, 0.2113732546567917, -0.3707558810710907, -0.7672045826911926, 0.25293904542922974, -1.1745219230651855, 0.798669159412384, -0.39910995960235596, 0.4275895357131958, 0.7743192315101624, 0.0825619176030159, 0.2618238925933838, -0.5989574790000916, -0.9975655674934387, 0.8394566178321838, -2.1975128650665283, -1.2507461309432983, -1.0535889863967896, 1.597631573677063, 0.8164443969726562, 0.1884782761335373, -2.1657960414886475, -1.0365771055221558, 2.306112766265869, 1.716174840927124, 0.0876004621386528, -0.9702048897743225, 0.3792857527732849, 0.14482277631759644, -2.4897947311401367, -0.04460381716489792, 0.11279446631669998, -0.9751885533332825, 1.098001480102539, 0.08420729637145996, 0.36186835169792175, -0.01966722123324871, -0.7612468600273132, -0.17357315123081207, 0.3223937153816223, 0.4815593361854553, 0.17077358067035675, -0.579607367515564, 0.08138953894376755, 1.4709488153457642, -1.5076299905776978, 1.5064642429351807, -0.3634166419506073, -0.09824015945196152, -0.18561393022537231, 1.453623652458191, 0.5478325486183167, 2.1248958110809326, 0.38536667823791504, 0.9942312240600586, -0.19258928298950195, 2.0332555770874023, -0.6886731386184692, -1.1863764524459839, -0.9027441740036011, -2.161803722381592, -1.5796231031417847, -1.4350428581237793, 0.25687944889068604, 0.32241788506507874, -0.5533194541931152, 2.406036615371704, 0.23253655433654785, 0.6761090755462646, -0.9134057760238647, -0.4411587417125702, -0.33520039916038513, -0.24083268642425537, 1.5302209854125977, -0.15313270688056946, 0.787310004234314, -0.16612663865089417, -0.3821852505207062, -0.18114282190799713, -0.6747902035713196, -0.4945361912250519, -0.9649360179901123, 1.0464000701904297, 0.30168935656547546, 0.7311753034591675, 0.8301510214805603, 0.8237370848655701, 0.009193355217576027, -0.09563937783241272, 0.8847682476043701, -1.4735822677612305, 0.1423988938331604, -0.6969307661056519, -1.052383542060852, 0.27746397256851196, 2.1099936962127686, -0.4881014823913574, -0.9466131329536438, 0.21484769880771637, -0.45158395171165466, 0.9034808874130249, -0.2423483431339264, 1.731109857559204, 0.14496105909347534, -1.8382766246795654, -0.12618328630924225, -0.46972528100013733, 0.21286025643348694, 0.45824918150901794, -0.4886411726474762, -1.9806123971939087, 0.1543022096157074, -1.4855756759643555, 0.5566175580024719, -0.4953063428401947, 0.23208606243133545, 1.4362396001815796, -0.9369887113571167, 1.9171457290649414, -1.0265898704528809, 0.5487058758735657, -1.14381742477417, 1.0341801643371582, -0.39005452394485474, 0.6658101081848145, -0.32712239027023315, -0.0026817095931619406, 0.9371163845062256, 0.34733232855796814, 0.615451991558075, 0.1695098727941513, -0.24219776690006256, -2.2494595050811768, -0.24639292061328888, 0.9720943570137024, 1.7702791690826416, -0.5317919850349426, 1.2099499702453613, -0.0946788489818573, -0.15132486820220947, 0.7144132256507874, 1.0372334718704224, 1.7555822134017944, -0.6537531614303589, 0.703174889087677, -0.6476990580558777, 0.5370593667030334, 0.8740666508674622, 1.2728297710418701, 0.2509717643260956, 1.0122652053833008, 1.0665355920791626, -0.11655319482088089, -0.06188998743891716, -0.14792747795581818, -0.944000780582428, -0.3543195128440857, -0.28708723187446594, -0.043628305196762085, 0.2126193642616272, 0.48616355657577515, -0.5799076557159424, -0.21766963601112366, 0.6989126801490784, 0.48903727531433105, -1.2673068046569824, 0.5284426808357239, 2.0728631019592285, -1.1514644622802734, -0.8450656533241272, 1.4057492017745972, 0.7378392219543457, -0.7873468995094299, -0.3865712583065033, -0.930840253829956, -0.05087185651063919, 0.3807719647884369, -0.23634234070777893, 0.9139230251312256, -0.6447957754135132, 1.416642189025879, 0.0655430480837822, -0.8657503128051758, 0.24566176533699036, 0.7157366871833801, -0.1863527148962021, 0.5175442099571228, -1.4624792337417603, -1.0944002866744995, -0.35929328203201294, 1.603407382965088, -0.20365752279758453, 0.7572015523910522, 1.5152446031570435, 0.14613422751426697, -0.29748570919036865, 0.10455143451690674, -0.6928504705429077, -0.6352468132972717, 1.5348039865493774, 0.06265688687562943, 0.5753762125968933, 0.8353413343429565, -0.10483114421367645, 1.1613284349441528, -1.8775163888931274, -0.2851978540420532, -0.8270783424377441, -0.1654365062713623, 0.27227625250816345, -0.06263323128223419, -0.8186404705047607, 1.8194782733917236, 0.4206022620201111, 1.353743314743042, 0.27008363604545593, -0.20160646736621857, 0.6579009294509888, -0.7437047958374023, 0.1802038997411728, -0.7038679718971252, 0.899929940700531, -0.11854676902294159, -0.8017587065696716, 0.7457811832427979, 0.12161274999380112, 0.6567474603652954, -1.0414488315582275, 0.2968510091304779, 1.329412579536438, 0.15673886239528656, 0.214753657579422, -0.872149646282196, -0.21566973626613617, 0.5280889868736267, -1.5802783966064453, 0.9198464751243591, -0.2916407287120819, 1.613323450088501, -1.4337655305862427, -1.0511294603347778, -0.4609123170375824, 0.6592576503753662, -0.17448055744171143, 0.422065407037735, -0.45378753542900085, 0.8988981246948242, 0.4650559425354004, 0.3843763768672943, 1.799668788909912, -0.0057244631461799145, 2.000795841217041, 1.3251335620880127, 1.4163001775741577, -0.9769514799118042, 0.39588120579719543, 0.47354328632354736, -1.0947600603103638, 1.053707480430603, -0.9399840831756592, -0.9127014875411987, 1.835397720336914, -1.0693024396896362, -0.7270243167877197, -0.9411932826042175, 0.16721870005130768, 0.48076266050338745, -1.1975834369659424, -0.9543222188949585, 0.9482883214950562, -0.2638278305530548, 0.14271937310695648, 0.2670290172100067, 0.6080020666122437, -0.6030563116073608, -0.7021837830543518, -0.912387490272522, -1.2669609785079956, 0.9593220949172974, 0.8385506272315979, -1.1314142942428589, 0.8271214365959167, 2.4973533153533936, 0.6534172296524048, -0.514674961566925, -0.9373206496238708, -0.16603532433509827, -1.5338693857192993, 0.03437340632081032, -2.1697349548339844, -1.0827099084854126, -0.33324670791625977, 0.8458532094955444, 1.9214860200881958, 0.4752958118915558, -1.119974970817566, 0.02809438668191433, 1.9592180252075195, -1.323162317276001, -1.62931489944458, 0.4008947014808655, -0.037290554493665695, 0.7444465756416321, -0.6441229581832886, -0.3801989257335663, -1.210093379020691, -0.7535220384597778, 0.5864384174346924, 0.7877315878868103, -2.1228325366973877, -0.6980457901954651, 1.2097322940826416, 1.2209739685058594, 0.4916526973247528, -0.43827277421951294, -0.09375090152025223, -0.22171279788017273, 1.6252086162567139, -1.2940887212753296, 0.7193577289581299, 0.8801977634429932, -0.4026007652282715, -0.7631155252456665, 0.20112910866737366, -0.0565946027636528, -0.6424749493598938, 0.6593754887580872, 0.05174344405531883, -0.7051172852516174, -0.6239987015724182, -0.27223116159439087, 1.0425993204116821, -0.279512494802475, -0.30633044242858887, 2.0203495025634766, -0.1547006070613861, 0.4410155415534973, -1.3363182544708252, 1.3676011562347412, 0.6126486659049988, -0.10253583639860153, 0.43005380034446716, -1.4292008876800537, -1.7676571607589722, 0.41948702931404114, 1.122771978378296, 0.5651838779449463, 0.3782930374145508, -1.6714988946914673, -0.466070294380188, -2.2727816104888916, 0.5583868026733398, -0.9366220235824585, 0.63246089220047, 1.2936736345291138, -1.609505534172058, -0.13996079564094543, -0.7028713226318359, -0.1058158427476883, 0.7829306125640869, -0.30490168929100037, 0.03795768693089485, 0.07993430644273758, -0.7239088416099548, -0.8406633138656616, 0.9556929469108582, 0.0885569378733635, -1.3756908178329468, 0.23202407360076904, 1.1923059225082397, -0.2681322693824768, -0.33659785985946655, -0.9834603667259216, -1.054287314414978, -0.014173685573041439, -0.7006771564483643, 0.009759224951267242, -1.641980767250061, 0.9246442317962646, 0.2528248131275177, 1.0642234086990356, 0.06608390808105469, -0.3375743627548218, 0.41953983902931213, -0.9783626794815063, -0.3588276207447052, -1.8346881866455078, 1.74476957321167, 1.5266239643096924, 1.326817274093628, -0.6174735426902771, 0.36036303639411926, -0.4114395081996918, 1.1689555644989014, -0.4390966296195984, -0.3801621198654175, -0.1395334154367447, 0.5983375906944275, 0.6594043970108032, 0.9599189162254333, 1.5878304243087769, 1.0753719806671143, -0.16991308331489563, 1.1624724864959717, -0.6652770042419434, 0.9807502627372742, 0.3347114622592926, 0.8693361282348633, 0.07763875275850296, -0.5926822423934937, -1.3930331468582153, 0.09841270744800568, 1.0871996879577637, -0.94852614402771, 0.9377344250679016, 1.8551030158996582, -1.3480666875839233, -0.20170235633850098, -0.4105105996131897, 0.17409665882587433, 0.12891672551631927, -0.24262212216854095, -0.08306678384542465, -2.480329751968384, -1.309238314628601, -0.07418385148048401, 0.7491716742515564, -0.13243621587753296, -0.8998346924781799, 1.5077464580535889, -0.14738473296165466, -0.45954960584640503, 0.7049882411956787, 0.27829501032829285, -1.9580450057983398, 0.6471636295318604, 0.9516172409057617, -1.726054310798645, -0.3531058728694916, 0.38531750440597534, -0.39086616039276123, -0.7920107841491699, 0.23561891913414001, 2.0890886783599854, 0.17939543724060059, 1.1463443040847778, 0.8589257597923279, -0.5417303442955017, -0.8307878375053406, -1.0272140502929688, 0.7273796796798706, -1.7352051734924316, 0.9205018281936646, 0.3829992711544037, 1.3512468338012695, 0.9375742077827454, -0.05929708108305931, -0.5590407252311707, 0.679498553276062, -0.13073191046714783, -0.7604880332946777, 1.472626805305481, 0.2052462100982666, 0.49470651149749756, 0.21355795860290527, -0.9062966108322144, -0.5432236194610596, -0.6944561004638672, 0.09291527420282364, -1.460656762123108, 0.49675172567367554, -0.6897757053375244, 0.053393665701150894, -0.03857478126883507, 1.4956374168395996, 1.8703737258911133, -1.0923713445663452, 1.6172773838043213, -1.4993330240249634, 1.3378936052322388, -1.887368083000183, -0.8172678351402283, 0.17159295082092285, -1.3643699884414673, 1.1372066736221313, -0.24542096257209778, -0.19891305267810822, -0.5959442853927612, -0.6551329493522644, -1.9280674457550049, 0.6862985491752625, -0.10910331457853317, 2.16778564453125, -1.4260748624801636, 0.6274083852767944, -2.2417163848876953, -0.8712560534477234, 0.9770851135253906, 1.5036062002182007, 1.591065526008606, -0.5565973520278931, -1.571832299232483, 1.0634241104125977, -2.5557427406311035, 1.485175371170044, 0.4212546944618225, 0.3176228702068329, -1.0101568698883057, 0.4418216645717621, -1.308954119682312, -1.5303223133087158, -2.1477627754211426, -1.7532219886779785, -0.6342665553092957, 1.283748745918274, -1.8351601362228394, 0.6350611448287964, 1.2789233922958374, 0.4730518162250519, -0.9004794359207153, 0.7220864295959473, -0.09494023770093918, 1.1155279874801636, 0.6158807277679443, -0.3414609134197235, 0.8314565420150757, 0.8496711850166321, 1.8975484371185303, -0.0009466403280384839, -0.5276891589164734, -0.6797686219215393, 1.0002821683883667, -0.4586564004421234, 0.0026424145326018333, 0.4451031982898712, -1.9109618663787842, -0.15216794610023499, 1.5109862089157104, 1.0981242656707764, 0.3323728144168854, 0.23327867686748505, -0.2880231440067291, -0.3058253526687622, -0.08492573350667953, 0.8909150958061218, 0.34573832154273987, 0.6078673601150513, -0.4807971715927124, 0.6012436747550964, 0.058517567813396454, 1.4429951906204224, -0.2568519413471222, -0.4466075897216797, -0.15260636806488037, 0.5329762101173401, -0.560986340045929, 1.02604341506958, 1.0595133304595947, 0.5104148387908936, 0.06475774198770523, -0.5403203964233398, 0.8274520635604858, 1.286371111869812, -0.12541931867599487, -1.2030048370361328, -1.3017910718917847, -0.039418648928403854, 0.4596036374568939, 0.779621422290802, 0.02721564471721649, -0.30799487233161926, 0.19753065705299377, -0.6983176469802856, -0.4921022653579712, 1.797879934310913, -1.2263787984848022, -0.36160004138946533, -0.7158343195915222, -1.385800838470459, -0.5164952874183655, 0.8501858711242676, -1.2303576469421387, -0.17138560116291046, -1.9224598407745361, 0.13879746198654175, 0.7341115474700928, 0.5712445974349976, 0.17759712040424347, -0.1393864005804062, 0.29873427748680115, -0.18961991369724274, -1.1321747303009033, -0.9707309603691101, -0.12452645599842072, 0.6756948828697205, -1.1475002765655518, -1.5439887046813965, 0.03588567674160004, 1.0037271976470947, 0.2978333830833435, 0.15069904923439026, 0.5793206095695496, 1.6180639266967773, 2.0207223892211914, 0.6140375733375549, -0.9594742655754089, -0.686916708946228, -0.8771212697029114, -0.48461541533470154, -2.4849541187286377, -0.4441658556461334, 0.1331106722354889, -1.492623209953308, -0.25555238127708435, 0.02642744779586792, -1.9630343914031982, -0.5383783578872681, 2.539036989212036, 0.10794847458600998, 0.40037983655929565, 0.5438440442085266, -0.5242874026298523, -0.6457220911979675, -0.3351050019264221, -0.22774910926818848, -0.49961331486701965, 0.40133950114250183, -1.0532087087631226, -0.2371196746826172, -0.5639227032661438, 0.651087760925293, -0.7620949149131775, 0.07393510639667511, -0.3973359763622284, -0.7483199834823608, 0.12127869576215744, 0.22489400207996368, 0.3376378118991852, -2.3980302810668945, -1.391999363899231, -0.8030433654785156, 0.753506064414978, -0.29362234473228455, 0.7513962388038635, 0.21961450576782227, -1.0463876724243164, -1.6735659837722778, 1.331702470779419, -0.07813020050525665, 0.07605768740177155, -0.26379162073135376, 0.8269366025924683, 0.043818335980176926, -0.8470582365989685, -0.5430693626403809, -1.7018523216247559, 2.0164711475372314, 1.8224037885665894, -0.7286966443061829, 1.4379321336746216, -0.1982315480709076, -1.0345162153244019, -0.08686650544404984, 0.6009531021118164, -0.0865618959069252, -1.1475379467010498, -2.5344760417938232, -1.2009427547454834, 1.7807846069335938, -0.588803231716156, 1.2491637468338013, 0.29745617508888245, 0.2885867655277252, 2.3174781799316406, 0.7066724300384521, -1.2678648233413696, 0.9456925988197327, 0.09392107278108597, 0.10290030390024185, 0.5394082069396973, 0.988559901714325, -2.742018222808838, 1.5738427639007568, 2.379979133605957, 0.04006391018629074, 1.7595171928405762, -0.7252795100212097, -0.8914439678192139, 0.08411110192537308, -2.473496437072754, -0.28602972626686096, 1.023252010345459, -0.7757314443588257, 0.06909739971160889, 0.19069625437259674, 1.486573338508606, 1.0650019645690918, 0.5378958582878113, 0.6025598049163818, 0.8358518481254578, -0.7980788350105286, 0.19821995496749878, 0.6107640266418457, 0.6906680464744568, -0.5576740503311157, -1.9790204763412476, 0.12122266739606857, -0.531383752822876, 0.2294088900089264, 0.47447025775909424, -1.4995075464248657, 0.048900824040174484, -0.058261897414922714, -0.05970709025859833, -0.4286932945251465, -0.515457034111023, 0.7523190975189209, -0.7547479867935181, -0.9388329982757568, 0.8767147064208984, -1.6990561485290527, -0.301992803812027, -0.5008063912391663, 0.9280923008918762, 0.643855631351471, 0.76854008436203, -1.6257386207580566, -0.2930702269077301, 0.9274925589561462, -0.5100335478782654, 0.3967115581035614, 1.0319061279296875, 0.9066326022148132, -0.1633436381816864, 1.7775176763534546, 0.32324856519699097, -1.0128190517425537, 0.7080858945846558, 0.7540293335914612, 1.7160828113555908, 0.1008632555603981, 0.18121202290058136, 1.3331798315048218, -0.3810480833053589, 0.43879956007003784, 0.13215231895446777, 0.49228760600090027, -1.253422737121582, 0.5793288350105286, -0.0859963521361351, -0.11727508902549744, -1.2880312204360962, -0.6459041237831116, -0.20782092213630676, 0.6224981546401978, -0.32686513662338257, -0.8733187317848206, 0.23924985527992249, 1.8393549919128418, -1.1363177299499512, 0.5830305218696594, -0.9756665825843811, -0.05331549420952797, 1.6123319864273071, 0.4234444499015808, -0.14985348284244537, -0.8761427998542786, 0.39618420600891113, 0.0834280326962471, 1.1740984916687012, -1.0393503904342651, 0.8874646425247192, 0.2711770236492157, 1.8542814254760742, -0.9062606692314148, 0.8719813823699951, 1.457385778427124, -0.5413382053375244, 0.6367203593254089, -1.4373505115509033, 0.4217972159385681, 2.0714943408966064, 1.556269645690918, 0.39505550265312195, -1.7254061698913574, -0.5853987336158752, -1.7510607242584229, -1.0753310918807983, -0.24329665303230286, 0.6115108728408813, -0.6383802890777588, -1.1324745416641235, -0.9877833127975464, 0.12691718339920044, 1.749651551246643, 0.7756463885307312, 0.9526108503341675, -0.26905378699302673, -2.0845232009887695, -0.6992999911308289, 0.015800664201378822, -0.07671581208705902, 0.7080976366996765, -1.0418988466262817, -0.7264387607574463, -1.4912619590759277, -0.1175365000963211, -0.04266168177127838, -0.19976060092449188, 0.8960088491439819, 0.1541859209537506, -0.2723933160305023, 0.6217518448829651, -2.5969274044036865, -1.097446322441101, -0.5203141570091248, -0.16346192359924316, 0.9035511612892151, 1.3341031074523926, 1.1261543035507202, -0.8285493850708008, 1.2355058193206787, 0.7653201818466187, 0.4662286043167114, 0.2721670866012573, -1.400826334953308, 0.22699129581451416, 0.2318216860294342, -0.7199422717094421, 0.5007291436195374, -0.6042752265930176, -0.01765967532992363, 1.6302236318588257, -0.7209822535514832, 0.900496244430542, -0.5663363933563232, -0.3459247350692749, -1.4964309930801392, -0.9008095264434814, 0.7201826572418213, -0.9302109479904175, -0.4671117067337036, -0.017684536054730415, -1.8644071817398071, -0.37312015891075134, 0.517450213432312, -1.9341949224472046, 0.11615186929702759, 1.2728023529052734, 0.7650904059410095, -0.1548997461795807, 0.7529006600379944, 0.40496626496315, -0.5659409165382385, 2.003265619277954, -0.947007417678833, 0.6561878323554993, 0.0655735656619072, 1.2199769020080566, 0.3746030032634735, 1.346311092376709, 2.5796327590942383, 0.8647646307945251, 0.12286427617073059, -0.3450677990913391, -1.9564779996871948, -0.10551497340202332, 0.3584651052951813, -0.2049187272787094, 0.7716960310935974, 1.7877285480499268, -0.7635708451271057, 0.014709985814988613, -2.0660557746887207, 0.25611743330955505, -0.33511725068092346, -0.30319494009017944, 0.3023918569087982, -0.4639210104942322, 2.0045652389526367, -2.1086463928222656, 0.7038505673408508, 0.7319813370704651, -0.8515975475311279, 2.1982641220092773, 0.12577860057353973, 0.45075902342796326, -1.1184182167053223, 1.6713390350341797, -0.535914421081543, -0.763047456741333, -2.8284242153167725, 0.3686659634113312, -0.607268214225769, -0.15342077612876892, -0.9964504241943359, 2.060946464538574, -1.410832405090332, -1.8886897563934326, -0.11044581234455109, -0.043389447033405304, 0.6938410997390747, -0.021286681294441223, 0.8431775569915771, 1.3096901178359985, 1.2894240617752075, 0.787492573261261, 0.4788166880607605, 0.983920693397522, 2.8116037845611572, -1.9529987573623657, -0.8592854738235474, 0.7251381874084473, -1.9661961793899536, 0.7430130243301392, -0.27779078483581543, 0.05404597520828247, -1.3724361658096313, -0.3866216540336609, 0.8728654980659485, 1.591970443725586, -0.6394312977790833, 0.06368488073348999, 0.034878674894571304, 0.04248122498393059, -0.05212055519223213, 0.5165233612060547, -1.2245715856552124, -1.4272350072860718, 0.4751487374305725, 1.071542739868164, 1.1949716806411743, 1.3998091220855713, 1.5200915336608887, -1.4166532754898071, -0.7414400577545166, -0.7327038049697876, -0.6529632806777954, 0.9379454255104065, 0.5848538875579834, 0.10938876122236252, -0.2285979837179184, -1.3575400114059448, 0.7357901930809021, 0.9218006134033203, -1.8328691720962524, -0.5952087640762329, -2.2963359355926514, -0.6495880484580994, 1.1621205806732178, 0.6923332214355469, -1.06865656375885, 1.9467966556549072, 1.0918750762939453, 0.5013550519943237, -1.0295450687408447, 1.1447455883026123, 0.4519761800765991, -0.9621619582176208, -0.41851434111595154, 0.08624488115310669, -0.7078466415405273, 1.2080833911895752, -1.0884435176849365, 0.2786150276660919, 0.2130705714225769, -0.6313477754592896, 0.2874658703804016, 0.4221736788749695, 0.570292592048645, -0.38138705492019653, 0.0837387815117836, 0.2472478449344635, -0.7628324627876282, 0.014393321238458157, 0.8283596038818359, 2.059053659439087, -0.8209401369094849, -0.6004120707511902, -1.5710171461105347, -1.6579993963241577, 0.23019452393054962, 1.1559749841690063, 0.9288116693496704, 0.07038433104753494, -1.0679006576538086, 1.9324736595153809, -0.10673375427722931, -0.21868716180324554, -0.8987632989883423, 1.131714940071106, 1.3736292123794556, -0.0017777391476556659, -0.24359366297721863, 0.5009456872940063, 1.5349140167236328, 0.8180439472198486, -0.28708580136299133, -0.23855282366275787, 0.26263707876205444, 1.162591576576233, 1.682823896408081, 1.17756187915802, -0.08104640990495682, 0.07251828163862228, -0.4253369867801666, -0.574028730392456, -0.8692253828048706, 1.3616905212402344, -1.0647820234298706, -0.00870580691844225, -0.41655462980270386, 2.049528121948242, -2.5646939277648926, 0.355146586894989, -0.25696131587028503, 0.4355427622795105, -0.31600064039230347, 2.8797333240509033, -0.21249501407146454, 0.32515886425971985, 1.3482633829116821, -1.3272849321365356, -1.1183605194091797, 2.025639772415161, -0.42443645000457764, 0.8693031668663025, 0.06146383285522461, -0.3951925039291382, -0.7100589275360107, 1.6611002683639526, 1.8317482471466064, 0.4176924526691437, 0.5315325260162354, -0.5393456816673279, 1.2732808589935303, 0.5327905416488647, 1.424359917640686, -0.5934875011444092, 2.1716175079345703, 1.3584964275360107, -0.7632532119750977, 2.1586594581604004, -0.8836745619773865, 0.12816645205020905, -0.567889928817749, -1.2581977844238281, 0.1676684021949768, -0.3430735766887665, -2.0291223526000977, 0.21963056921958923, 0.454511821269989, 0.7924857139587402, 0.13337136805057526, 1.184516429901123, -0.032063744962215424, 0.29442811012268066, -0.485365092754364, -0.7591651082038879, -1.5076956748962402, 0.3440585732460022, 0.2646278738975525, -0.5645043253898621, -1.0493634939193726, -0.5857744216918945, -0.544622540473938, 0.7599837183952332, -0.32430338859558105, -0.09993236511945724, 1.6832711696624756, 1.0582420825958252, 0.25272735953330994, 1.43523108959198, -0.5564504861831665, -0.6668736338615417, -0.8822989463806152, -1.3016921281814575, -0.707843005657196, -0.013537665829062462, -0.07385092973709106, 0.3771597146987915, 0.20108471810817719, 1.6501455307006836, -2.0577828884124756, -0.21571557223796844, 0.5460643172264099, -0.6901816129684448, -0.10071740299463272, 0.2197386920452118, -0.4103958308696747, 1.2428001165390015, 0.3706875145435333, -2.015458583831787, 0.3396519720554352, -1.4702739715576172, -2.129276990890503, 0.18033145368099213, -0.5425726771354675, 0.26807132363319397, 0.01509188488125801, -0.8740342855453491, -0.3011505901813507, -1.3488795757293701, 0.39983755350112915, 0.19368088245391846, 1.343305230140686, -1.5059008598327637, 0.5861009955406189, 0.4316682815551758, 0.5190325975418091, 1.2228567600250244, -1.0264812707901, -0.6296202540397644, -0.8652029633522034, -0.8441370725631714, -0.8125873804092407, 1.1856348514556885, -0.222963348031044, 0.5664573907852173, 0.4854215383529663, -0.3332241177558899, -0.40746745467185974, -0.47601982951164246, 0.3603501319885254, -0.22345148026943207, 0.07497747987508774, -0.8324414491653442, 1.2504016160964966, 0.32205286622047424, -0.032608844339847565, -0.3875664174556732, 0.42356398701667786, -0.27141299843788147, -1.192967176437378, -1.1262016296386719, 1.0492490530014038, -1.509242296218872, 1.0440551042556763, -0.3526522219181061, -1.017721176147461, 0.3086327612400055, -0.04487564042210579, -0.002685045124962926, 1.8377703428268433, -0.6020447611808777, 1.329810619354248, -0.39052239060401917, 0.29617661237716675, 1.2178133726119995, 0.036661360412836075, 0.7698361277580261, 0.7135682702064514, -1.0059490203857422, -1.471665620803833, 1.2007439136505127, -0.12810534238815308, -0.9277978539466858, 1.5322890281677246, 0.5921419262886047, -0.010901869274675846, 1.5060023069381714, 1.179603099822998, 0.0712561383843422, -0.4242720901966095, -0.5528494119644165, -0.2781868875026703, -1.1619268655776978, 1.2992942333221436, -0.1166037991642952, -0.09305747598409653, 0.7116669416427612, -0.2008245289325714, -2.024578332901001, -1.2510265111923218, 0.9796746969223022, 1.3000866174697876, -0.5462669134140015, -0.44407469034194946, -0.06661491841077805, -2.2245304584503174, 0.831307053565979, 0.005064641125500202, -0.4415936768054962, -0.0012193182483315468, -0.4464752972126007, 0.13276219367980957, 0.5216853618621826, -0.6931025981903076, -0.6341830492019653, 0.09776253998279572, 1.034114956855774, 0.39820560812950134, -0.8082599639892578, 0.4754306674003601, 0.17814476788043976, 0.4786742925643921, -0.6807187795639038, 0.611508846282959, 0.24230434000492096, 0.5763734579086304, 0.3116042912006378, 1.8836497068405151, 0.5662328004837036, 0.1783466637134552, -0.5344176888465881, -0.06489035487174988, 1.2533732652664185, 0.8630638122558594, 1.661774754524231, 0.19592000544071198, 1.2532042264938354, 0.2848769426345825, -0.40131041407585144, 0.4215445816516876, 1.7169513702392578, -0.903565526008606, -0.02672388404607773, 0.6438657641410828, 0.16824883222579956, 0.6323282718658447, 2.609858512878418, 0.13050758838653564, 0.26217615604400635, -0.2985014319419861, 0.23651158809661865, -0.1717667281627655, 1.3208692073822021, 2.2631306648254395, -0.3308585286140442, -0.7580779790878296, -0.8275174498558044, -0.3195589780807495, -0.24108295142650604, 0.5754519701004028, -0.28847014904022217, -1.778115153312683, -0.19227761030197144, -0.229854017496109, 0.6199237704277039, -0.07432655990123749, -1.4157893657684326, -0.29436635971069336, -1.2491990327835083, -0.2759290635585785, 0.326105535030365, -0.9004629850387573, -0.983863890171051, -0.9256193041801453, -0.8285049796104431, 1.2919427156448364, -1.0884648561477661, -1.4400626420974731, -0.6541271805763245, 0.34287354350090027, -0.39952781796455383, -1.233367919921875, -1.4429395198822021, -0.13938023149967194, -0.47225359082221985, -0.11579863727092743, -0.5523926019668579, 1.1315438747406006, -0.744509220123291, -0.16504397988319397, -0.07757813483476639, -0.33555394411087036, 0.14567776024341583, 0.6454870700836182, 0.15139517188072205, 0.13353697955608368, 0.3582857847213745, 1.0009260177612305, -0.5363940000534058, -0.11644872277975082, 1.06761634349823, 0.7112115025520325, 1.9496326446533203, -0.19610925018787384, -0.9767947196960449, 1.4879072904586792, -1.1961902379989624, -0.50959712266922, 0.9618427157402039, -0.16807666420936584, 0.13486655056476593, 0.21548974514007568, 0.3903595507144928, -1.87167227268219, -0.9589319229125977, 1.0034171342849731, 0.8048433065414429, -0.21241946518421173, -0.9576407670974731, -0.9660550951957703, -0.19402280449867249, -0.4538048803806305, -1.4358254671096802, -0.33773910999298096, -0.07987817376852036, 0.1776733547449112, -0.4969281554222107, -0.3560545742511749, 1.008069396018982, -0.6336752772331238, -0.8802067637443542, -0.27094390988349915, -0.6552992463111877, 0.20374608039855957, 1.532951831817627, 0.3294394612312317, -1.0373355150222778, -0.29430216550827026, -0.9071391224861145, -0.9977226257324219, 1.0937387943267822, -0.8196894526481628, -1.0461320877075195, 0.26942694187164307, 0.5026610493659973, 1.2379999160766602, 1.560226559638977, -1.7810639142990112, 0.9503692388534546, 1.0308529138565063, 1.62046217918396, -0.15517909824848175, -1.7752904891967773, -0.623673141002655, -0.3244423270225525, -0.7387359738349915, -0.2085256725549698, 1.3425581455230713, -0.8104985356330872, -0.47746041417121887, 1.7966452836990356, -0.06856177002191544, -1.2992017269134521, 1.225152611732483, -0.37406039237976074, -0.017737818881869316, -1.008704662322998, 0.035543765872716904, 0.0015045769978314638, 0.9240682721138, 0.23511894047260284, -0.5549296736717224, -0.2539141774177551, 1.897302508354187, 0.5425488352775574, -1.0669784545898438, -0.3308783173561096, 0.468106746673584, 0.6509608030319214, -0.5735706090927124, 0.23043033480644226, -0.8277888894081116, -0.5915812849998474, 0.25743022561073303, 0.9794437289237976, -0.06954033672809601, 0.35834988951683044, -0.6088696718215942, 0.45818471908569336, -1.5488464832305908, -0.8732175827026367, 0.05846808850765228, 0.0008078320533968508, -1.1538233757019043, 0.24912498891353607, 0.25277477502822876, 0.4648529887199402, -0.3517376780509949, 1.3157140016555786, -0.08533963561058044, -0.8102124929428101, -0.09199123829603195, -0.7213879227638245, -0.48847657442092896, -1.3020278215408325, -0.6409683227539062, 0.07237496227025986, -0.32662272453308105, -0.20746305584907532, -2.3951575756073, 0.5042951107025146, -0.11163483560085297, 1.452614188194275, 0.35253605246543884, 2.0167202949523926, 0.5044494271278381, -0.2013688087463379, 0.16592878103256226, 0.06883088499307632, 0.09024239331483841, 0.5482707023620605, -1.7557488679885864, 0.18136067688465118, -0.5203869342803955, 1.322576642036438, -1.0561585426330566, -1.4738807678222656, -0.3514111042022705, 0.7445765733718872, -0.27893730998039246, 0.38161781430244446, 0.4272350072860718, -1.3943281173706055, -0.5811841487884521, -0.4902665317058563, 1.1940232515335083, -2.4498260021209717, -0.026743097230792046, 0.7228229641914368, 1.908096432685852, -1.6375126838684082, -2.937103509902954, -0.24314481019973755, 0.2039133906364441, -2.7141411304473877, -0.1970944106578827, -0.01882912777364254, 0.479428768157959, -0.5504897832870483, 0.7415487766265869, -2.228053569793701, 0.12207986414432526, -1.0973695516586304, 0.1744978427886963, -0.37532174587249756, 1.0456469058990479, -0.7681070566177368, -1.4192321300506592, -1.433837890625, -0.4590704143047333, 0.2858707904815674, 1.3632667064666748, 0.2684013843536377, 0.7949010133743286, 0.6217046976089478, -0.6695716381072998, -0.5294691324234009, 0.5773894786834717, -1.4281928539276123, 1.4474403858184814, -0.5128531455993652, 0.03191550076007843, -1.6687382459640503, 2.1006641387939453, -0.8612935543060303, 0.48178988695144653, -1.2385990619659424, 0.36225807666778564, 0.02428004890680313, 0.8267439603805542, -2.2753641605377197, 0.4535514712333679, -1.6339970827102661, 0.8110350966453552, -0.8955824375152588, -0.2365189492702484, 0.5181331038475037, -0.3124955892562866, 0.44505149126052856, -0.7588384747505188, -0.2401910126209259, -0.8761773109436035, 1.5429071187973022, -2.179736614227295, 2.289684534072876, 1.839761734008789, -0.3081400692462921, -0.059080056846141815, 0.8708460330963135, 0.7655038833618164, -0.4352799355983734, 2.8721494674682617, -0.23993553221225739, -1.0966286659240723, 0.8522946834564209, -0.1383415311574936, 0.3696807622909546, -0.01986132562160492, -0.9387653470039368, 0.6652434468269348, -0.09429202973842621, -0.775048553943634, 0.36385807394981384, 0.4785290062427521, 0.7226316332817078, -1.128733515739441, -0.3797379732131958, -1.0514366626739502, -1.034659743309021, -0.5888211131095886, 0.9269682765007019, 0.014070702716708183, 0.13501854240894318, 0.1640181690454483, 0.8958443999290466, 0.3027927875518799, 1.396979808807373, 0.7774153351783752, 0.7450788021087646, 1.7334717512130737, 1.7601985931396484, -0.04688447713851929, 1.1407426595687866, -1.124220371246338, -1.4031108617782593, -0.45874181389808655, -0.018554633483290672, 0.7787273526191711, -1.3528287410736084, -0.8988345861434937, 1.466149926185608, -0.5189083218574524, 1.1544712781906128, 1.1369045972824097, 1.589576244354248, 0.33779585361480713, -0.4586629569530487, 0.4716412425041199, -0.15922363102436066, -1.6285223960876465, -0.3760080337524414, 0.00014232919784262776, 0.4323737919330597, 0.5607861876487732, -0.25707492232322693, -0.9300318360328674, 0.5248075723648071, -0.7532084584236145, 0.5182875394821167, -0.5045914053916931, 2.3132402896881104, 2.1357944011688232, -0.3146633207798004, -0.5126360058784485, 0.38635313510894775, -0.3292824923992157, -0.755912721157074, 2.890568256378174, 1.2653604745864868, -0.06146543473005295, 0.293797105550766, -0.12672151625156403, 0.21024291217327118, -0.6108394265174866, 0.16050754487514496, 0.19367863237857819, 0.6954331994056702, 1.4644124507904053, -0.31766802072525024, 0.6775103211402893, 0.9975491166114807, -1.9329818487167358, 0.11747374385595322, -0.08298291265964508, 0.3902841806411743, -0.29554882645606995, -0.4715075194835663, -1.3276219367980957, 1.893471598625183, 1.5827546119689941, -0.9588305354118347, -0.3168473243713379, 0.39939066767692566, -1.4189457893371582, 0.5912340879440308, 0.4455101191997528, -0.1468649059534073, -0.9562296867370605, 0.002449852181598544, 0.4390875995159149, 1.871991753578186, -0.6224202513694763, -1.4390504360198975, -0.6465574502944946, 0.34561148285865784, -0.003670146455988288, -0.18684057891368866, 0.4804123342037201, 0.4576362669467926, 0.4854622483253479, -0.8356838226318359, -0.37604427337646484, -0.6441569924354553, 0.7175530195236206, 0.10432291775941849, -0.06882419437170029, 0.6472970247268677, -0.3649646043777466, -0.9893835186958313, 0.26106032729148865, 0.8173368573188782, 0.45691388845443726, 1.1321958303451538, 1.162926197052002, 0.08452358841896057, -0.9438576698303223, 0.46871820092201233, -2.3805315494537354, 0.9277063012123108, -1.236409068107605, -0.24756643176078796, 0.44029563665390015, 0.27950119972229004, 0.13783015310764313, -0.9368519186973572, -1.1229667663574219, -0.8185805082321167, 0.2524472773075104, 0.3280281126499176, 0.004913709592074156, 0.43999937176704407, -1.0955514907836914, -0.21561604738235474, 0.7506684064865112, 0.3359646201133728, -0.6228429079055786, -1.3131070137023926, 0.2078104466199875, 0.21119244396686554, 1.4907822608947754, 2.6777899265289307, 1.1228549480438232, -2.421520709991455, 0.8498672842979431, -0.1418372541666031, -0.8436132073402405, -1.527958631515503, 0.05712847784161568, -0.04591383785009384, -0.05316122993826866, 0.24815762042999268, 0.6439266204833984, -0.8489117622375488, 0.33835968375205994, -0.7736186385154724, 0.36671745777130127, -0.04238641634583473, 1.0583990812301636, 1.3552427291870117, 0.6554275155067444, 1.6801220178604126, 0.923870325088501, -2.1378378868103027, 1.063554048538208, 0.24796022474765778, -1.2453404664993286, 1.2940210103988647, -0.5644067525863647, 1.413036823272705, 0.24719321727752686, 0.23265624046325684, -1.0557500123977661, -0.6143066883087158, 1.3624320030212402, -0.18769407272338867, -0.5311805605888367, -0.7989357113838196, 0.7727325558662415, -1.1251006126403809, -0.6804470419883728, -0.4188818335533142, -0.05776797980070114, -0.4753324091434479, 0.015064261853694916, -1.2672119140625, -0.1860276609659195, -0.5643026232719421, -0.14710690081119537, 0.13862523436546326, -0.33179402351379395, -0.5997695326805115, 0.8117744326591492, -1.7020883560180664, 0.07716026157140732, -0.08404803276062012, 0.9578226804733276, -0.023693816736340523, 0.5641934275627136, -1.7179744243621826, -0.984355628490448, -0.7952588796615601, 1.9320573806762695, 0.15876179933547974, 0.13697433471679688, 0.027555331587791443, -1.7850276231765747, 0.09772215038537979, 0.09750813245773315, 0.2649823725223541, -2.6337060928344727, -0.3961460590362549, -0.24677638709545135, -0.26861944794654846, -0.9745138883590698, 0.8776845335960388, 1.9372193813323975, -1.0582764148712158, 0.7418795228004456, -1.5501160621643066, -0.12446673214435577, 1.7438055276870728, 1.2616108655929565, 1.5219727754592896, 0.7105839252471924, -1.147125244140625, 0.9681876301765442, -2.410667657852173, -0.6261901259422302, 0.13281607627868652, 0.8979112505912781, -0.6183185577392578, 1.3262945413589478, 1.4999727010726929, 2.253617286682129, 0.5597596168518066, -0.289302796125412, -0.17144307494163513, -0.4361419081687927, 1.866471767425537, -1.7566688060760498, -0.7040368914604187, 2.1605396270751953, 0.44080135226249695, -0.27265849709510803, -1.119101881980896, 1.1513080596923828, -0.15760986506938934, -1.5127062797546387, 0.15324270725250244, 0.7569069862365723, -1.7756818532943726, -1.0621744394302368, 0.3615442216396332, -0.3186313211917877, 0.5029577612876892, -0.31241482496261597, 0.24850335717201233, -1.0410374402999878, 1.622065544128418, 0.21858452260494232, 0.761254072189331, -1.4735485315322876, -1.111868143081665, -0.9797124862670898, -0.5470806956291199, -0.5076447129249573, -1.204576015472412, 1.531491994857788, 0.390289843082428, 0.7732358574867249, 0.38152387738227844, 1.7397867441177368, -1.2175167798995972, 1.5067079067230225, 1.3485965728759766, -1.3893399238586426, 1.3501261472702026, -1.8405134677886963, 0.5138311386108398, -0.07393566519021988, -0.5886120200157166, 0.9775946140289307, 0.471230149269104, 0.28502655029296875, -0.24078074097633362, 2.132208824157715, -0.032476767897605896, -0.7004827857017517, -2.328676700592041, 1.037811279296875, 2.306246757507324, 0.8989907503128052, -1.525638461112976, -1.266060709953308, -0.4626213610172272, -1.6489580869674683, -0.34893813729286194, 1.9121272563934326, -3.0700793266296387, 0.9899991154670715, -1.1016361713409424, 0.7313740253448486, 1.2674939632415771, 1.2425884008407593, -2.463144063949585, 2.6198039054870605, -0.5922062397003174, -0.38317304849624634, 0.7349918484687805, -0.07408396154642105, 0.22107501327991486, 0.4493826925754547, -0.33432477712631226, -0.0011797471670433879, 1.525627613067627, 0.65937340259552, -0.2908850312232971, -0.33342570066452026, -0.49379444122314453, -1.3728045225143433, 1.555809736251831, 0.9915028214454651, -0.4912094175815582, 0.12626942992210388, 1.7359402179718018, -0.6390640139579773, 1.2430320978164673, -1.2011178731918335, -0.47519204020500183, -0.03665228560566902, 0.5054935812950134, -1.3832906484603882, -0.5543696880340576, 0.13996656239032745, -1.057973861694336, -0.6455017924308777, 0.5969361662864685, -0.14694471657276154, -0.3620704412460327, 0.8856307864189148, -0.9925574660301208, 0.7286067008972168, 1.2789397239685059, 1.0268291234970093, 0.11833617836236954, 2.014268159866333, -1.9336823225021362, -1.208376169204712, -0.47417786717414856, 0.3693910837173462, -1.2505617141723633, 1.6175521612167358, -1.6815431118011475, 1.5616015195846558, -0.23729364573955536, -0.09509041905403137, -0.3849795162677765, 2.8605587482452393, 0.5098214149475098, -1.7701725959777832, 1.2416391372680664, -1.223258137702942, 0.5739368796348572, 1.6990818977355957, -1.2427031993865967, 0.02717251516878605, 1.7074371576309204, 1.064127802848816, 0.796278715133667, -0.2692832052707672, -0.14545567333698273, 0.8932425379753113, -0.72733473777771, 0.4404856562614441, 0.2891014814376831, -0.8754238486289978, 1.081876516342163, -0.07658123970031738, 1.078322410583496, -1.618940830230713, -0.9734457731246948, 0.934438943862915, -0.10655481368303299, -0.15152762830257416, -0.3544013500213623, -0.12285950779914856, -0.7423831224441528, 0.04700833931565285, 0.029642170295119286, -0.7653011083602905, -0.46540889143943787, -1.5887843370437622, 0.3756885528564453, -0.7777593731880188, 0.4660276174545288, 0.2660931646823883, -0.6389275193214417, -0.22205610573291779, -0.4294207692146301, -0.14158833026885986, -1.8741711378097534, 1.28622305393219, 0.12373100966215134, 0.3818630874156952, -0.4734167158603668, 0.8818404674530029, -0.18425922095775604, 1.4495091438293457, -0.9085552096366882, 0.13463671505451202, -0.7932092547416687, -1.751121997833252, 1.7362136840820312, 0.24437271058559418, 0.04523443058133125, 0.2237119972705841, -0.2907058000564575, -1.4539767503738403, 0.4373578727245331, -1.4174268245697021, -1.8738664388656616, 1.2632766962051392, -0.615784764289856, -1.0572208166122437, 1.2461367845535278, -1.1132317781448364, -1.119283676147461, -0.679549515247345, -1.528430700302124, -0.3100348114967346, 0.7257022261619568, -0.9799674153327942, 1.0529468059539795, -0.46835052967071533, 1.6187623739242554, 1.0875705480575562, -0.7059224843978882, -0.16094033420085907, -1.1607497930526733, 0.7204174995422363, 0.05063493549823761, -0.09970380365848541, -0.8710658550262451, 0.5975975394248962, -2.1296160221099854, -2.23047137260437, -1.1303038597106934, 0.5266028046607971, -2.2449779510498047, 2.1754844188690186, 1.2164603471755981, -0.5296003222465515, 1.1927343606948853, 0.007493825629353523, 0.6562552452087402, 1.627852439880371, 1.430884838104248, 1.976666808128357, 0.23036164045333862, 1.1025108098983765, -0.2722766399383545, -0.2626776695251465, 0.8396421670913696, -0.9536986351013184, -1.0042479038238525, 0.15866944193840027, -0.13240377604961395, 0.027409620583057404, -0.4968276023864746, 1.1559274196624756, 2.3837201595306396, 1.5313438177108765, 0.58430415391922, 0.8520053625106812, -1.1606464385986328, 0.7087525129318237, 0.7909078001976013, 0.3465156555175781, -1.2290666103363037, -1.4960806369781494, -0.297393798828125, -0.351960152387619, 0.2643985152244568], "dtype": "float32"}], "e": 64.82319641113281}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(dot, json.loads(_TESTS), 'cuda', *[0.0001, 0.0001])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")